In [1]:
cd ..

/home/adrian/git_repos/recsys_content


# Reto 2 - SISTEMA DE RECOMENDACIÓN BASADA EN CONTENIDO

$$
rating = f(Usuario, Local, Interacción_{U-L}, Contexto)
$$

In [2]:
from pathlib import Path

import numpy as np
import polars as pl

In [3]:
DATA_DIR = Path("./data")
RESULTS_DIR = Path("./results")
RESULTS_DIR.mkdir(exist_ok=True)
TRAIN_FILE = DATA_DIR / "train_reviews.csv"
TEST_FILE = DATA_DIR / "test_reviews.csv"
BUSINESS_FILE = DATA_DIR / "negocios.csv"
USER_FILE = DATA_DIR / "usuarios.csv"
OUTPUT_FILE = RESULTS_DIR / "submission.csv"

## 1. Carga de datos

In [4]:
COUNTER_COLS = ["useful", "funny", "cool"]

train = pl.read_csv(TRAIN_FILE, infer_schema=20_000).with_columns(
    [pl.col(c).clip(lower_bound=0) for c in COUNTER_COLS]).with_columns(
    pl.col("date").str.strptime(pl.Datetime, "%Y-%m-%d %H:%M:%S").alias("datetime"),
).with_columns(
    pl.col("datetime").dt.date().alias("date"))
test = pl.read_csv(TEST_FILE, infer_schema=20_000).with_columns(
    pl.col("date").str.strptime(pl.Datetime, "%Y-%m-%d %H:%M:%S").alias("datetime"),
).with_columns(
    pl.col("datetime").dt.date().alias("date"))


display(train.head())

review_id,user_id,business_id,stars,useful,funny,cool,date,datetime
str,str,str,f64,i64,i64,i64,date,datetime[μs]
"""ZZO43qKB-s65zplC8RfJqw""","""-1BSu2dt_rOAqllw9ZDXtA""","""smkZq4G1AOm4V6p3id5sww""",5.0,0,0,0,2016-09-30,2016-09-30 15:49:32
"""vojXOF_VOgvuKD95gCO8_Q""","""xpe178ng_gj5X6HgqtOing""","""96_c_7twb7hYRZ9HHrq01g""",1.0,2,0,1,2020-12-09,2020-12-09 14:39:51
"""KwxdbiseRlIRNzpgvyjY0Q""","""axbaerf2Fk92OB4b9_peVA""","""e0AYjKfSF0DL-5C1CpOq6Q""",4.0,0,0,0,2013-09-04,2013-09-04 16:19:51
"""3mwoBcTy-2gMh0L91uaIeA""","""_GOiybb0rImYKJfwyxEaGg""","""vF-uptiQ34pVLHJKzPHUlA""",5.0,0,0,0,2019-03-02,2019-03-02 12:24:14
"""XfWf7XsBWs3kYyYq7Ns1ZQ""","""ojWKg3B5pH3ncAsxun3kUw""","""X28XK71RuEXPapeyUOwNzg""",5.0,10,4,7,2020-04-23,2020-04-23 18:26:29


In [5]:
businesses = pl.read_csv(BUSINESS_FILE, infer_schema=20_000)
display(businesses.head(20))

business_id,name,address,city,state,postal_code,latitude,longitude,stars,review_count,is_open,attributes,categories,hours
str,str,str,str,str,str,f64,f64,f64,i64,i64,str,str,str
"""GDEEPQdYs2utMN-R4znZSA""","""Metro Self Storage - Largo""","""10501 S Belcher Rd""","""Largo""","""FL""","""33777""",27.868519,-82.743849,4.5,7,0,"""{'BusinessAcceptsCreditCards':…","""Packing Supplies, Shopping, Lo…","""{'Monday': '0:0-0:0', 'Tuesday…"
"""pbAq2NRG_2jCBI6fgRalvQ""","""Madewell""","""3301 Veterans Blvd, Ste 84""","""Metairie""","""LA""","""70002""",30.005894,-90.15745,2.0,6,1,"""{'BusinessAcceptsCreditCards':…","""Shopping, Women's Clothing, Fa…","""{'Monday': '10:0-21:0', 'Tuesd…"
"""h4HP3Vc0dQq7SfSLal9qQw""","""Dollylocks""","""511 9th St N""","""Saint Petersburg""","""FL""","""33701""",27.777785,-82.646673,4.5,7,0,"""{'RestaurantsPriceRange2': '4'…","""Hair Stylists, Beauty & Spas, …",null
"""PndbFVbHE4730HDlghxv6g""","""Jim Browne Chrysler Jeep Dodge…","""10909 N Florida Ave""","""Tampa""","""FL""","""33612""",28.048093,-82.458757,2.5,76,1,"""{'BusinessAcceptsCreditCards':…","""Automotive, Auto Parts & Suppl…","""{'Monday': '7:30-20:0', 'Tuesd…"
"""IayDnngl0NooAbcoo62j-w""","""Emg Salons""","""324 S Falkenburg Rd""","""Tampa""","""FL""","""33619""",27.948815,-82.334619,4.5,7,1,"""{'WiFi': ""u'free'"", 'BikeParki…","""Hair Stylists, Hair Salons, Be…","""{'Tuesday': '9:0-20:0', 'Wedne…"
…,…,…,…,…,…,…,…,…,…,…,…,…,…
"""VUe_7JxmIauXFRxFuurCdQ""","""The Woodhouse Day Spa - New Or…","""4030 Canal St""","""New Orleans""","""LA""","""70119""",29.97399,-90.100419,4.0,195,1,"""{'BusinessParking': ""{'garage'…","""Day Spas, Skin Care, Massage, …","""{'Monday': '10:0-19:0', 'Tuesd…"
"""D6_nRMZwMxTr1Kvqv7hbNw""","""Grace Pilates + Yoga""","""4223 Magazine St""","""New Orleans""","""LA""","""70115""",29.921113,-90.09975,5.0,5,1,"""{'BusinessParking': ""{'garage'…","""Pilates, Yoga, Fitness & Instr…","""{'Monday': '6:0-20:0', 'Tuesda…"
"""1l0BxtHnqb4anMI34VfshA""","""Pabbys Pet Resort And Day Care""","""101 Stewart Ln""","""Chalfont""","""PA""","""18914""",40.283436,-75.242029,3.0,13,1,null,"""Pet Sitting, Pet Services, Pet…","""{'Monday': '7:0-19:0', 'Tuesda…"


In [6]:
users = pl.read_csv(USER_FILE, infer_schema=20_000)
display(users.head())

user_id,name,review_count,yelping_since,useful,funny,cool,elite,friends,fans,average_stars,compliment_hot,compliment_more,compliment_profile,compliment_cute,compliment_list,compliment_note,compliment_plain,compliment_cool,compliment_funny,compliment_writer,compliment_photos
str,str,i64,str,i64,i64,i64,str,str,i64,f64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64
"""qVc8ODYU5SZjKXVBgXdI7w""","""Walker""",585,"""2007-01-25 16:47:26""",7217,1259,5994,"""2007""","""NSCy54eWehBJyZdG2iE84w, pe42u7…",267,3.91,250,65,55,56,18,232,844,467,467,239,180
"""j14WgRoU_-2ZE1aw1dXrJg""","""Daniel""",4333,"""2009-01-25 04:35:42""",43091,13066,27281,"""2009,2010,2011,2012,2013,2014,…","""ueRPE0CX75ePGMqOFVj6IQ, 52oH4D…",3138,3.74,1145,264,184,157,251,1847,7054,3131,3131,1521,1946
"""2WnXYQFK0hXEoTxPtV2zvg""","""Steph""",665,"""2008-07-25 10:41:00""",2086,1010,1003,"""2009,2010,2011,2012,2013""","""LuO3Bn4f3rlhyHIaNfTlnA, j9B4Xd…",52,3.32,89,13,10,17,3,66,96,119,119,35,18
"""SZDeASXq7o05mMNLshsdIA""","""Gwen""",224,"""2005-11-29 04:38:33""",512,330,299,"""2009,2010,2011""","""enx1vVPnfdNUdPho6PH_wg, 4wOcvM…",28,4.27,24,4,1,6,2,12,16,26,26,10,9
"""q_QQ5kBBwlCcbL1s4NVK3g""","""Jane""",1221,"""2005-03-14 20:26:35""",14953,9940,11211,"""2006,2007,2008,2009,2010,2011,…","""xBDpTUbai0DXrvxCe3X16Q, 7GPNBO…",1357,3.85,1713,163,191,361,147,1212,5696,2543,2543,815,323


## 2. Creación de los datos de entrada - Offline

### 2.1 Usuario

In [7]:
from datetime import date
import math
REFERENCE_DATE = date(2023, 1, 1)

COUNTER_COLS = ["useful", "funny", "cool"]

COMPLIMENT_COLS = [
    "compliment_hot",
    "compliment_more",
    "compliment_profile",
    "compliment_cute",
    "compliment_list",
    "compliment_note",
    "compliment_plain",
    "compliment_cool",
    "compliment_funny",
    "compliment_writer",
    "compliment_photos",
]

In [8]:
def _safe_log10(expr: pl.Expr) -> pl.Expr:
    return (
        expr.fill_null(0)
        .clip(lower_bound=0)
        .add(1)
        .log10()
        .fill_null(0)
        .fill_nan(0)
    )


def _csv_count(expr: pl.Expr) -> pl.Expr:
    return (
        pl.when(expr.is_null() | (expr == ""))
        .then(0)
        .otherwise(expr.str.split(",").list.len())
        .cast(pl.Int32)
    )


In [9]:
import polars as pl
import math

def build_user_features_batched(users: pl.DataFrame, batch_size: int = 50000) -> pl.DataFrame:
    # 1. Variables globales necesarias para mantener la integridad de la red
    # Calculamos la media global con todo el dataset
    global_mean_avg_stars = users.select(pl.col("average_stars").mean()).item()
    
    # Creamos un diccionario ligero de amigos global para que los joins no devuelvan nulos
    global_friend_profiles = users.select(
        pl.col("user_id").alias("friend_id"),
        pl.col("average_stars").alias("friend_average_stars"),
    )

    # 2. Función interna que procesa un solo batch
    def process_batch(batch: pl.DataFrame) -> pl.DataFrame:
        base = (
            batch.select(
                "user_id",
                "name",
                "review_count",
                "yelping_since",
                "useful",
                "funny",
                "cool",
                "elite",
                "friends",
                "fans",
                "average_stars",
                *COMPLIMENT_COLS,
            )
            .with_columns(
                pl.col("yelping_since")
                .str.strptime(pl.Datetime, "%Y-%m-%d %H:%M:%S", strict=False)
                .alias("yelping_since"),
                *[pl.col(c).clip(lower_bound=0).fill_null(0) for c in COUNTER_COLS],
                *[pl.col(c).fill_null(0) for c in COMPLIMENT_COLS],
                pl.col("friends")
                .fill_null("")
                .str.split(",")
                .list.eval(pl.element().str.strip_chars())
                .alias("friends_list"),
                pl.col("elite")
                .fill_null("")
                .str.split(",")
                .list.eval(pl.element().str.strip_chars())
                .alias("elite_list"),
            )
            .with_columns(
                # Calendario
                pl.col("yelping_since").dt.year().alias("signup_year"),
                pl.col("yelping_since").dt.month().alias("signup_month"),
                pl.col("yelping_since").dt.quarter().alias("signup_quarter"),
                pl.col("yelping_since").dt.weekday().alias("signup_weekday"),
                pl.col("yelping_since").dt.ordinal_day().alias("signup_day_of_year"),
                pl.col("yelping_since").dt.day().alias("signup_day_of_month"),
                pl.col("yelping_since").dt.week().alias("signup_week_of_year"),
                pl.col("yelping_since").dt.hour().alias("signup_hour"),
                (pl.col("yelping_since").dt.weekday() >= 6).cast(pl.Int8).alias("signup_is_weekend"),
                (pl.col("yelping_since").dt.weekday() < 6).cast(pl.Int8).alias("signup_is_workday"),
                (pl.col("yelping_since").dt.day() <= 3).cast(pl.Int8).alias("signup_is_month_start"),
                (pl.col("yelping_since").dt.day() >= 28).cast(pl.Int8).alias("signup_is_month_end"),
                pl.when(pl.col("yelping_since").dt.month().is_in([12, 1, 2])).then(0)
                .when(pl.col("yelping_since").dt.month().is_in([3, 4, 5])).then(1)
                .when(pl.col("yelping_since").dt.month().is_in([6, 7, 8])).then(2)
                .otherwise(3)
                .cast(pl.Int8)
                .alias("signup_season_id"),
                (pl.col("friends") != "").cast(pl.Int8).alias("has_friends"),
                (pl.col("elite") != "").cast(pl.Int8).alias("is_elite"),
                _csv_count(pl.col("friends")).alias("friends_count"),
                _csv_count(pl.col("elite")).alias("elite_count"),
                ((pl.lit(REFERENCE_DATE) - pl.col("yelping_since").dt.date()).dt.total_days()).alias("account_age_days"),
            )
            .with_columns(
                (pl.col("account_age_days") / 365.25).alias("years_active"),
                pl.when(pl.col("account_age_days") < 365).then(0)
                .when(pl.col("account_age_days") < 365 * 3).then(1)
                .when(pl.col("account_age_days") < 365 * 7).then(2)
                .otherwise(3)
                .cast(pl.Int8)
                .alias("account_age_bucket_id"),
                # Codificación cíclica
                (2 * math.pi * (pl.col("signup_month") - 1) / 12).sin().alias("signup_month_sin"),
                (2 * math.pi * (pl.col("signup_month") - 1) / 12).cos().alias("signup_month_cos"),
                (2 * math.pi * (pl.col("signup_weekday") - 1) / 7).sin().alias("signup_weekday_sin"),
                (2 * math.pi * (pl.col("signup_weekday") - 1) / 7).cos().alias("signup_weekday_cos"),
                (2 * math.pi * (pl.col("signup_day_of_year") - 1) / 366).sin().alias("signup_doy_sin"),
                (2 * math.pi * (pl.col("signup_day_of_year") - 1) / 366).cos().alias("signup_doy_cos"),
            )
        )

        user_profiles = base.select(
            "user_id",
            pl.col("average_stars").alias("user_average_stars"),
        )

        # Aquí hacemos join contra el `global_friend_profiles` para no perder datos de otros batches
        friend_edges = (
            base.select("user_id", "friends_list")
            .explode("friends_list")
            .rename({"friends_list": "friend_id"})
            .filter(
                pl.col("friend_id").is_not_null()
                & (pl.col("friend_id") != "")
                & (pl.col("friend_id") != pl.col("user_id"))
            )
            .join(user_profiles, on="user_id", how="left")
            .join(global_friend_profiles, on="friend_id", how="left")
        )

        friend_stats = (
            friend_edges.group_by("user_id")
            .agg(
                pl.len().alias("friends_edge_count"),
                pl.col("friend_id").n_unique().alias("friends_unique_count"),
                pl.col("friend_average_stars").count().alias("friends_valid_profile_count"),
                pl.col("friend_average_stars").mean().alias("friends_avg_stars_mean"),
                pl.col("friend_average_stars").std().alias("friends_avg_stars_std"),
                pl.col("friend_average_stars").min().alias("friends_avg_stars_min"),
                pl.col("friend_average_stars").max().alias("friends_avg_stars_max"),
                pl.col("friend_average_stars").median().alias("friends_avg_stars_median"),
                pl.col("friend_average_stars").quantile(0.25, interpolation="linear").alias("friends_avg_stars_q25"),
                pl.col("friend_average_stars").quantile(0.75, interpolation="linear").alias("friends_avg_stars_q75"),
                (pl.col("friend_average_stars") > global_mean_avg_stars).mean().alias("friends_above_global_ratio"),
                (pl.col("friend_average_stars") < global_mean_avg_stars).mean().alias("friends_below_global_ratio"),
                pl.col("friend_average_stars").is_null().mean().alias("friends_missing_profile_ratio"),
                (pl.col("friend_average_stars") - pl.col("user_average_stars")).mean().alias("friend_minus_user_stars_gap_mean"),
                (pl.col("friend_average_stars") - pl.col("user_average_stars")).std().alias("friend_minus_user_stars_gap_std"),
                (pl.col("friend_average_stars") - pl.col("user_average_stars")).abs().mean().alias("friend_abs_user_gap_mean"),
                (pl.col("friend_average_stars") > pl.col("user_average_stars")).mean().alias("friends_above_user_ratio"),
                (pl.col("friend_average_stars") < pl.col("user_average_stars")).mean().alias("friends_below_user_ratio"),
            )
            .with_columns(
                (pl.col("friends_avg_stars_q75") - pl.col("friends_avg_stars_q25")).alias("friends_avg_stars_iqr"),
                (pl.col("friends_avg_stars_max") - pl.col("friends_avg_stars_min")).alias("friends_avg_stars_range"),
                (
                    pl.col("friends_avg_stars_std")
                    / (pl.col("friends_avg_stars_mean") + 1e-6)
                ).alias("friends_avg_stars_cv"),
                (pl.col("friends_valid_profile_count") / (pl.col("friends_edge_count") + 1e-6)).alias("friend_coverage_ratio"),
                (pl.col("friends_edge_count") - pl.col("friends_unique_count")).alias("friend_duplicate_count"),
                (pl.col("friends_edge_count") - pl.col("friends_valid_profile_count")).alias("friend_missing_profile_count"),
            )
        )

        df_batch = (
            base.join(friend_stats, on="user_id", how="left")
            .with_columns(
                pl.col("friends_edge_count").fill_null(0),
                pl.col("friends_unique_count").fill_null(0),
                pl.col("friends_valid_profile_count").fill_null(0),
                pl.col("friends_avg_stars_mean").fill_null(global_mean_avg_stars),
                pl.col("friends_avg_stars_std").fill_null(0),
                pl.col("friends_avg_stars_min").fill_null(global_mean_avg_stars),
                pl.col("friends_avg_stars_max").fill_null(global_mean_avg_stars),
                pl.col("friends_avg_stars_median").fill_null(global_mean_avg_stars),
                pl.col("friends_avg_stars_q25").fill_null(global_mean_avg_stars),
                pl.col("friends_avg_stars_q75").fill_null(global_mean_avg_stars),
                pl.col("friends_avg_stars_iqr").fill_null(0),
                pl.col("friends_avg_stars_range").fill_null(0),
                pl.col("friends_avg_stars_cv").fill_null(0),
                pl.col("friends_above_global_ratio").fill_null(0),
                pl.col("friends_below_global_ratio").fill_null(0),
                pl.col("friends_missing_profile_ratio").fill_null(0),
                pl.col("friend_coverage_ratio").fill_null(0),
                pl.col("friend_duplicate_count").fill_null(0),
                pl.col("friend_missing_profile_count").fill_null(0),
                pl.col("friend_minus_user_stars_gap_mean").fill_null(0),
                pl.col("friend_minus_user_stars_gap_std").fill_null(0),
                pl.col("friend_abs_user_gap_mean").fill_null(0),
                pl.col("friends_above_user_ratio").fill_null(0),
                pl.col("friends_below_user_ratio").fill_null(0),
                (pl.col("average_stars") - pl.col("friends_avg_stars_mean")).alias("user_friend_stars_gap"),
                (pl.col("average_stars") - pl.col("friends_avg_stars_median")).alias("user_friend_median_gap"),
                (pl.col("friends_avg_stars_mean") - global_mean_avg_stars).alias("friends_reputation_bias"),
            )
            .with_columns(
                _safe_log10(pl.col("review_count")).alias("review_count_log"),
                _safe_log10(pl.col("fans")).alias("fans_log"),
                _safe_log10(pl.col("useful")).alias("useful_log"),
                _safe_log10(pl.col("funny")).alias("funny_log"),
                _safe_log10(pl.col("cool")).alias("cool_log"),
                _safe_log10(pl.col("friends_count")).alias("friends_count_log"),
                _safe_log10(pl.col("elite_count")).alias("elite_count_log"),
                _safe_log10(pl.col("years_active")).alias("years_active_log"),
                _safe_log10(pl.col("account_age_days")).alias("account_age_days_log"),
                _safe_log10(pl.col("friends_edge_count")).alias("friends_edge_count_log"),
                _safe_log10(pl.col("friends_avg_stars_mean")).alias("friends_avg_stars_mean_log"),
                _safe_log10(pl.col("friends_avg_stars_std")).alias("friends_avg_stars_std_log"),
            )
            .with_columns(
                (pl.col("useful") / (pl.col("review_count") + 1)).alias("useful_per_review"),
                (pl.col("funny") / (pl.col("review_count") + 1)).alias("funny_per_review"),
                (pl.col("cool") / (pl.col("review_count") + 1)).alias("cool_per_review"),
                (pl.col("review_count") / (pl.col("years_active") + 1e-6)).alias("reviews_per_year"),
                (pl.col("fans") / (pl.col("years_active") + 1e-6)).alias("fans_per_year"),
                (pl.col("friends_count") / (pl.col("years_active") + 1e-6)).alias("friends_per_year"),
                (pl.col("elite_count") / (pl.col("years_active") + 1e-6)).alias("elite_per_year"),
                (pl.col("review_count") / (pl.col("account_age_days") + 1e-6)).alias("reviews_per_day"),
                (pl.col("fans") / (pl.col("account_age_days") + 1e-6)).alias("fans_per_day"),
                (pl.col("fans") / (pl.col("friends_count") + 1)).alias("fan_friend_ratio"),
                (pl.col("average_stars") - global_mean_avg_stars).alias("user_bias"),
            )
            .with_columns(
                pl.sum_horizontal([pl.col(c) for c in COMPLIMENT_COLS]).alias("total_compliments"),
                pl.sum_horizontal([(pl.col(c) > 0).cast(pl.Int16) for c in COMPLIMENT_COLS]).alias("compliment_nonzero_types"),
            )
            .with_columns(
                _safe_log10(pl.col("total_compliments")).alias("comp_total_log"),
                _safe_log10(pl.col("compliment_writer")).alias("comp_writer_log"),
                _safe_log10(pl.col("compliment_photos")).alias("comp_photos_log"),
                (pl.col("total_compliments") / (pl.col("review_count") + 1)).alias("compliments_per_review"),
                (pl.col("total_compliments") / (pl.col("years_active") + 1e-6)).alias("compliments_per_year"),
                (pl.col("compliment_nonzero_types") / len(COMPLIMENT_COLS)).alias("compliment_diversity"),
            )
            .with_columns(
                pl.max_horizontal([pl.col(c) for c in COMPLIMENT_COLS]).alias("compliment_max"),
                pl.min_horizontal([pl.col(c) for c in COMPLIMENT_COLS]).alias("compliment_min"),
                (pl.sum_horizontal([pl.col(c) for c in COMPLIMENT_COLS]) / len(COMPLIMENT_COLS)).alias("compliment_mean"),
            )
            .with_columns(
                (_safe_log10(pl.col("review_count")) + _safe_log10(pl.col("fans"))).alias("social_reach_score"),
                (_safe_log10(pl.col("review_count")) + pl.col("average_stars")).alias("expertise_proxy"),
                (_safe_log10(pl.col("fans") + 1) + pl.col("average_stars")).alias("visibility_proxy"),
                (_safe_log10(pl.col("review_count")) + _safe_log10(pl.col("total_compliments"))).alias("engagement_score"),
                (pl.col("average_stars") * _safe_log10(pl.col("review_count") + 1)).alias("rating_strength"),
                ((pl.col("useful") + pl.col("funny") + pl.col("cool")) / (pl.col("review_count") + 1)).alias("feedback_intensity"),
                (_safe_log10(pl.col("fans") + pl.col("friends_count") + pl.col("total_compliments"))).alias("popularity_mass_log"),
                (pl.col("user_bias") * pl.col("years_active")).alias("bias_x_age"),
                (pl.col("review_count_log") * pl.col("fans_log")).alias("activity_x_reach"),
                (pl.col("review_count_log") * pl.col("signup_is_weekend")).alias("activity_x_weekend_signup"),
                (pl.col("user_friend_stars_gap").abs()).alias("user_friend_stars_gap_abs"),
            )
            .with_columns(
                (pl.col("user_bias") ** 2).alias("user_bias_sq"),
                (pl.col("feedback_intensity") ** 2).alias("feedback_intensity_sq"),
                (pl.col("compliment_diversity") ** 2).alias("compliment_diversity_sq"),
                (pl.col("friends_avg_stars_cv") ** 2).alias("friends_avg_stars_cv_sq"),
            )
            .drop(
                "name",
                "yelping_since",
                "friends",
                "elite",
                "friends_list",
                "elite_list",
                *COMPLIMENT_COLS,
                "compliment_max",
                "compliment_min",
            )
        )
        return df_batch

    # 3. Iteramos por bloques dividiendo el dataframe y apilando los resultados
    processed_batches = []
    total_rows = users.height
    
    for offset in range(0, total_rows, batch_size):
        # Tomamos el bloque correspondiente
        batch = users.slice(offset, batch_size)
        # Procesamos y guardamos
        processed_batch = process_batch(batch)
        processed_batches.append(processed_batch)
        
    # 4. Concatenamos todo de vuelta
    return pl.concat(processed_batches, how="vertical")

In [10]:
users = build_user_features_batched(users)

In [11]:
users.head()

user_id,review_count,useful,funny,cool,fans,average_stars,signup_year,signup_month,signup_quarter,signup_weekday,signup_day_of_year,signup_day_of_month,signup_week_of_year,signup_hour,signup_is_weekend,signup_is_workday,signup_is_month_start,signup_is_month_end,signup_season_id,has_friends,is_elite,friends_count,elite_count,account_age_days,years_active,account_age_bucket_id,signup_month_sin,signup_month_cos,signup_weekday_sin,signup_weekday_cos,signup_doy_sin,signup_doy_cos,friends_edge_count,friends_unique_count,friends_valid_profile_count,friends_avg_stars_mean,…,friends_avg_stars_mean_log,friends_avg_stars_std_log,useful_per_review,funny_per_review,cool_per_review,reviews_per_year,fans_per_year,friends_per_year,elite_per_year,reviews_per_day,fans_per_day,fan_friend_ratio,user_bias,total_compliments,compliment_nonzero_types,comp_total_log,comp_writer_log,comp_photos_log,compliments_per_review,compliments_per_year,compliment_diversity,compliment_mean,social_reach_score,expertise_proxy,visibility_proxy,engagement_score,rating_strength,feedback_intensity,popularity_mass_log,bias_x_age,activity_x_reach,activity_x_weekend_signup,user_friend_stars_gap_abs,user_bias_sq,feedback_intensity_sq,compliment_diversity_sq,friends_avg_stars_cv_sq
str,i64,i64,i64,i64,i64,f64,i32,i8,i8,i8,i16,i8,i8,i8,i8,i8,i8,i8,i8,i8,i8,i32,i32,i64,f64,i8,f64,f64,f64,f64,f64,f64,u32,u32,u32,f64,…,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,i64,i16,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64
"""qVc8ODYU5SZjKXVBgXdI7w""",585,7217,1259,5994,267,3.91,2007,1,1,4,25,25,4,16,0,1,0,0,0,1,1,14995,1,5820,15.934292,3,0.0,1.0,0.433884,-0.900969,0.400454,0.916317,14995,14995,1821,3.814212,…,0.682525,0.12333,12.3157,2.148464,10.228669,36.713271,16.756313,941.052132,0.062758,0.100515,0.045876,0.017805,0.227471,2873,11,3.458487,2.380211,2.257679,4.90273,180.302953,1.0,261.181818,5.196032,6.677898,6.339752,6.226384,10.825375,24.692833,4.258542,3.624589,6.720829,0.0,0.095788,0.051743,609.73599,1.0,0.007413
"""j14WgRoU_-2ZE1aw1dXrJg""",4333,43091,13066,27281,3138,3.74,2009,1,1,7,25,25,4,4,1,0,0,0,0,1,1,4646,14,5089,13.932923,3,0.0,1.0,-0.781831,0.62349,0.400454,0.916317,4646,4646,1300,3.907269,…,0.69084,0.130838,9.942547,3.014767,6.294647,310.990005,225.221933,333.454781,1.004814,0.851444,0.616624,0.675274,0.057471,20631,11,4.314541,3.182415,3.289366,4.760268,1480.737318,1.0,1875.545455,7.13368,7.376889,7.23693,7.95143,13.602339,19.251961,4.453563,0.800739,12.717442,3.636889,0.167269,0.003303,370.638011,1.0,0.008096
"""2WnXYQFK0hXEoTxPtV2zvg""",665,2086,1010,1003,52,3.32,2008,7,3,5,207,25,30,10,0,1,0,0,2,1,1,381,5,5273,14.436687,3,1.2246e-16,-1.0,-0.433884,-0.900969,-0.384665,-0.923056,381,381,59,3.81322,…,0.682436,0.108946,3.132132,1.516517,1.506006,46.063196,3.601934,26.391094,0.34634,0.126114,0.009862,0.136126,-0.362529,585,11,2.767898,1.556303,1.278754,0.878378,40.521759,1.0,53.181818,4.54775,6.143474,5.052394,5.591372,9.376098,6.154655,3.008174,-5.233718,4.868448,0.0,0.49322,0.131427,37.879774,1.0,0.005591
"""SZDeASXq7o05mMNLshsdIA""",224,512,330,299,28,4.27,2005,11,4,2,333,29,48,4,0,1,0,1,3,1,1,131,3,6242,17.089665,3,-0.866025,0.5,0.781831,0.62349,-0.551102,0.834438,131,131,27,3.932222,…,0.693043,0.114466,2.275556,1.466667,1.328889,13.107337,1.638417,7.665451,0.175545,0.035886,0.004486,0.212121,0.587471,136,11,2.136721,1.041393,1.0,0.604444,7.958026,1.0,12.363636,3.814581,6.622183,5.747121,4.488903,10.052043,5.071111,2.471292,10.039683,3.439827,0.0,0.337778,0.345122,25.716168,1.0,0.005881
"""q_QQ5kBBwlCcbL1s4NVK3g""",1221,14953,9940,11211,1357,3.85,2005,3,1,1,73,14,11,20,0,1,0,0,1,1,1,5843,9,6502,17.801506,3,0.866025,0.5,0.0,1.0,0.944489,0.328542,5843,5843,1236,3.768422,…,0.678375,0.120461,12.236498,8.134206,9.174304,68.589699,76.229502,328.230641,0.505575,0.187788,0.208705,0.232204,0.167471,15707,11,4.196121,2.91169,2.510545,12.853519,882.341038,1.0,1427.909091,6.219971,6.937071,6.983219,7.283192,11.886592,29.545008,

### 2.2 Local

In [12]:
import ast
import math
import re
from datetime import date

import numpy as np
import polars as pl

REFERENCE_DATE = date(2023, 1, 1)

DAYS = ["Monday", "Tuesday", "Wednesday", "Thursday", "Friday", "Saturday", "Sunday"]
WEEKEND_DAYS = {"Saturday", "Sunday"}

ATTR_BOOL_KEYS = [
    "BusinessAcceptsCreditCards",
    "BikeParking",
    "WheelchairAccessible",
    "GoodForKids",
    "GoodForGroups",
    "RestaurantsReservations",
    "RestaurantsDelivery",
    "RestaurantsTakeOut",
    "OutdoorSeating",
    "HasTV",
    "Open24Hours",
    "ByAppointmentOnly",
    "AcceptsInsurance",
    "DriveThru",
    "Caters",
    "DogsAllowed",
    "HappyHour",
    "CoatCheck",
    "RestaurantsGoodForGroups",
    "RestaurantsTableService",
    "RestaurantsCounterService",
]

AMBIENCE_KEYS = [
    "romantic",
    "intimate",
    "classy",
    "hipster",
    "divey",
    "touristy",
    "trendy",
    "upscale",
    "casual",
]

MEAL_KEYS = [
    "breakfast",
    "brunch",
    "lunch",
    "dinner",
    "dessert",
    "latenight",
]

BEST_NIGHTS_KEYS = ["monday", "tuesday", "wednesday", "thursday", "friday", "saturday", "sunday"]
MUSIC_KEYS = ["dj", "background_music", "jukebox", "live", "video", "karaoke"]

CATEGORY_GROUPS = {
    "cat_food": ["food", "restaurants", "cafe", "cafes", "coffee", "tea", "bakeries", "desserts", "ice cream"],
    "cat_bars": ["bars", "pubs", "nightlife", "cocktail", "beer", "wine"],
    "cat_shopping": ["shopping", "fashion", "gifts", "bookstores", "jewelry", "shopping centers"],
    "cat_beauty_spa": ["beauty & spas", "hair", "hair salons", "nail salons", "massage", "skin care", "cosmetics"],
    "cat_health_fitness": ["health", "fitness", "gyms", "active life", "pilates", "yoga", "martial arts"],
    "cat_automotive": ["automotive", "auto", "car", "tires", "oil change", "body shops", "parts"],
    "cat_home_services": ["home services", "contractors", "electricians", "plumbing", "roofing", "landscaping", "real estate"],
    "cat_travel_hotels": ["hotels", "travel", "tours", "vacation rentals", "resorts"],
    "cat_nightlife": ["nightlife", "dance clubs", "music venues", "karaoke"],
    "cat_active_life": ["active life", "sports clubs", "recreation", "hiking", "outdoor"],
    "cat_pets": ["pets", "pet services", "pet sitting", "pet grooming", "pet stores"],
    "cat_professional": ["professional services", "legal", "accountants", "marketing", "consultants", "it services"],
    "cat_arts_entertainment": ["arts & entertainment", "museums", "theater", "cinema", "festivals", "venues"],
    "cat_finance": ["financial services", "banks", "insurance", "tax services", "investing"],
    "cat_education": ["education", "schools", "tutoring", "child care"],
    "cat_real_estate": ["real estate", "property management", "apartments"],
    "cat_local_services": ["local services", "laundromat", "shipping centers", "mobile phone repair", "couriers"],
    "cat_event_planning": ["event planning & services", "caterers", "party supplies", "venues", "photographers", "wedding planning"],
    "cat_religion": ["religious organizations", "churches", "synagogues", "mosques", "temples"],
}


def _safe_log10(expr: pl.Expr) -> pl.Expr:
    return (
        expr.cast(pl.Float64, strict=False)
        .fill_null(0)
        .clip(lower_bound=0)
        .add(1)
        .log10()
        .fill_null(0)
        .fill_nan(0)
    )


def _coerce_scalar(v):
    if isinstance(v, str):
        t = v.strip().strip("'\"")
        if t.lower() in {"true", "false"}:
            return t.lower() == "true"
        if t.lower() in {"none", "null", ""}:
            return None
        try:
            return int(t)
        except Exception:
            try:
                return float(t)
            except Exception:
                return t
    return v


def _normalize_obj(obj):
    if isinstance(obj, dict):
        return {str(k): _normalize_obj(v) for k, v in obj.items()}
    if isinstance(obj, list):
        return [_normalize_obj(v) for v in obj]
    return _coerce_scalar(obj)


def _parse_dictish(x):
    if x is None:
        return {}
    if isinstance(x, dict):
        return _normalize_obj(x)
    if not isinstance(x, str):
        return {}
    s = x.strip()
    if s == "" or s.lower() == "null":
        return {}
    s = s.replace("u'", "'").replace('u"', '"')
    try:
        obj = ast.literal_eval(s)
        obj = _normalize_obj(obj)
        return obj if isinstance(obj, dict) else {}
    except Exception:
        return {}


def _boolish(v):
    if isinstance(v, bool):
        return v
    if isinstance(v, (int, float)):
        return bool(v)
    if isinstance(v, str):
        t = v.strip().strip("'\"").lower()
        if t in {"true", "1", "yes", "free"}:
            return True
        if t in {"false", "0", "no", "none", "null"}:
            return False
    return None


def _lower_text(v):
    if v is None:
        return ""
    return str(v).strip().strip("'\"").lower()


def _parse_time_to_min(s):
    if not s:
        return None
    m = re.match(r"^\s*(\d{1,2}):(\d{1,2})\s*$", str(s))
    if not m:
        return None
    hh = int(m.group(1))
    mm = int(m.group(2))
    return hh * 60 + mm


def _csv_count(expr: pl.Expr) -> pl.Expr:
    return (
        pl.when(expr.is_null() | (expr == ""))
        .then(0)
        .otherwise(expr.str.split(",").list.len())
        .cast(pl.Int32)
    )


def _extract_hours_features(raw_hours):
    feats = {}
    for day in DAYS:
        feats[f"hours_{day.lower()}_open"] = 0
        feats[f"hours_{day.lower()}_duration_min"] = 0
        feats[f"hours_{day.lower()}_start_min"] = 0
        feats[f"hours_{day.lower()}_close_min"] = 0

    d = _parse_dictish(raw_hours)
    if not d:
        for k in [
            "hours_days_present_count",
            "hours_open_days_count",
            "hours_closed_days_count",
            "hours_missing_days_count",
            "hours_weekly_total_minutes",
            "hours_weekly_total_hours",
            "hours_weekday_total_minutes",
            "hours_weekend_total_minutes",
            "hours_weekday_mean_minutes",
            "hours_weekend_mean_minutes",
            "hours_weekday_std_minutes",
            "hours_weekend_std_minutes",
            "hours_mean_open_minutes",
            "hours_std_open_minutes",
            "hours_min_open_minutes",
            "hours_max_open_minutes",
            "hours_mean_close_minutes",
            "hours_std_close_minutes",
            "hours_earliest_open_minute",
            "hours_latest_close_minute",
            "hours_open_days_ratio",
            "hours_weekend_hours_ratio",
            "hours_weekday_hours_ratio",
            "hours_late_open_days_count",
            "hours_early_open_days_count",
            "hours_late_close_days_count",
            "hours_overnight_days_count",
            "hours_24h_days_count",
            "hours_regular_schedule_cv",
            "hours_has_weekend_open",
            "hours_open_all_days",
            "hours_consistent_hours_flag",
            "hours_weekday_open_days_count",
            "hours_weekend_open_days_count",
        ]:
            feats[k] = 0
        return feats

    durations = []
    starts = []
    closes = []
    weekday_durations = []
    weekend_durations = []
    overnight = 0
    late_open = 0
    early_open = 0
    late_close = 0
    open_days = 0
    weekend_open_days = 0
    weekday_open_days = 0
    full_day = 0

    for day in DAYS:
        val = d.get(day)
        if not val or str(val).strip() in {"0:0-0:0", "00:00-00:00"}:
            continue
        parts = str(val).split("-")
        if len(parts) != 2:
            continue

        start_min = _parse_time_to_min(parts[0])
        close_min = _parse_time_to_min(parts[1])
        if start_min is None or close_min is None:
            continue

        dur = close_min - start_min if close_min >= start_min else (24 * 60 - start_min + close_min)
        if dur <= 0:
            continue

        feats[f"hours_{day.lower()}_open"] = 1
        feats[f"hours_{day.lower()}_duration_min"] = int(dur)
        feats[f"hours_{day.lower()}_start_min"] = int(start_min)
        feats[f"hours_{day.lower()}_close_min"] = int(close_min)

        open_days += 1
        durations.append(dur)
        starts.append(start_min)
        closes.append(close_min)

        if day in WEEKEND_DAYS:
            weekend_open_days += 1
            weekend_durations.append(dur)
        else:
            weekday_open_days += 1
            weekday_durations.append(dur)

        if close_min < start_min:
            overnight += 1
        if start_min <= 8 * 60:
            early_open += 1
        if start_min >= 18 * 60:
            late_open += 1
        if close_min >= 22 * 60:
            late_close += 1
        if dur >= 23 * 60 + 30:
            full_day += 1

    durations_arr = np.array(durations, dtype=float) if durations else np.array([0.0])
    starts_arr = np.array(starts, dtype=float) if starts else np.array([0.0])
    closes_arr = np.array(closes, dtype=float) if closes else np.array([0.0])
    weekday_arr = np.array(weekday_durations, dtype=float) if weekday_durations else np.array([0.0])
    weekend_arr = np.array(weekend_durations, dtype=float) if weekend_durations else np.array([0.0])

    total_minutes = float(durations_arr.sum())
    total_weekday = float(weekday_arr.sum())
    total_weekend = float(weekend_arr.sum())
    mean_minutes = float(durations_arr.mean()) if open_days else 0.0
    std_minutes = float(durations_arr.std()) if open_days > 1 else 0.0
    mean_start = float(starts_arr.mean()) if open_days else 0.0
    mean_close = float(closes_arr.mean()) if open_days else 0.0

    feats.update({
        "hours_days_present_count": int(open_days),
        "hours_open_days_count": int(open_days),
        "hours_closed_days_count": int(7 - open_days),
        "hours_missing_days_count": int(7 - open_days),
        "hours_weekly_total_minutes": total_minutes,
        "hours_weekly_total_hours": total_minutes / 60.0,
        "hours_weekday_total_minutes": total_weekday,
        "hours_weekend_total_minutes": total_weekend,
        "hours_weekday_mean_minutes": float(weekday_arr.mean()) if weekday_open_days else 0.0,
        "hours_weekend_mean_minutes": float(weekend_arr.mean()) if weekend_open_days else 0.0,
        "hours_weekday_std_minutes": float(weekday_arr.std()) if weekday_open_days > 1 else 0.0,
        "hours_weekend_std_minutes": float(weekend_arr.std()) if weekend_open_days > 1 else 0.0,
        "hours_mean_open_minutes": mean_start,
        "hours_std_open_minutes": float(starts_arr.std()) if open_days > 1 else 0.0,
        "hours_min_open_minutes": float(starts_arr.min()) if open_days else 0.0,
        "hours_max_open_minutes": float(starts_arr.max()) if open_days else 0.0,
        "hours_mean_close_minutes": mean_close,
        "hours_std_close_minutes": float(closes_arr.std()) if open_days > 1 else 0.0,
        "hours_earliest_open_minute": float(starts_arr.min()) if open_days else 0.0,
        "hours_latest_close_minute": float(closes_arr.max()) if open_days else 0.0,
        "hours_open_days_ratio": open_days / 7.0,
        "hours_weekend_hours_ratio": total_weekend / (total_minutes + 1e-6),
        "hours_weekday_hours_ratio": total_weekday / (total_minutes + 1e-6),
        "hours_late_open_days_count": int(late_open),
        "hours_early_open_days_count": int(early_open),
        "hours_late_close_days_count": int(late_close),
        "hours_overnight_days_count": int(overnight),
        "hours_24h_days_count": int(full_day),
        "hours_regular_schedule_cv": float(std_minutes / (mean_minutes + 1e-6)) if open_days > 1 else 0.0,
        "hours_has_weekend_open": int(1 if weekend_open_days > 0 else 0),
        "hours_open_all_days": int(1 if open_days == 7 else 0),
        "hours_consistent_hours_flag": int(1 if (open_days > 1 and std_minutes <= 60) else 0),
        "hours_weekday_open_days_count": int(weekday_open_days),
        "hours_weekend_open_days_count": int(weekend_open_days),
    })
    return feats


def _extract_attributes_features(raw_attr):
    d = _parse_dictish(raw_attr)

    feats = {
        "attr_top_key_count": 0,
        "attr_nested_dict_count": 0,
        "attr_scalar_count": 0,
        "attr_bool_true_count": 0,
        "attr_bool_false_count": 0,
        "attr_string_count": 0,
        "attr_numeric_count": 0,
        "attr_family_count": 0,
        "attr_price_range": 0,
        "attr_has_price_range": 0,
        "attr_wifi_free": 0,
        "attr_wifi_paid": 0,
        "attr_wifi_no": 0,
        "attr_has_wifi": 0,
        "attr_parking_any": 0,
        "attr_parking_garage": 0,
        "attr_parking_street": 0,
        "attr_parking_lot": 0,
        "attr_parking_valet": 0,
        "attr_parking_validated": 0,
        "attr_parking_true_count": 0,
        "attr_has_business_parking": 0,
        "attr_has_ambience": 0,
        "attr_has_good_for_meal": 0,
        "attr_has_best_nights": 0,
        "attr_has_music": 0,
        "attr_alcohol_none": 0,
        "attr_alcohol_beer_and_wine": 0,
        "attr_alcohol_full_bar": 0,
        "attr_noise_low": 0,
        "attr_noise_average": 0,
        "attr_noise_loud": 0,
        "attr_noise_very_loud": 0,
        "attr_attire_casual": 0,
        "attr_attire_dressy": 0,
        "attr_attire_formal": 0,
        "attr_music_dj": 0,
        "attr_music_background_music": 0,
        "attr_music_jukebox": 0,
        "attr_music_live": 0,
        "attr_music_video": 0,
        "attr_music_karaoke": 0,
        "attr_music_true_count": 0,
        "attr_ambience_true_count": 0,
        "attr_meal_true_count": 0,
        "attr_best_nights_true_count": 0,
        "attr_attribute_richness": 0,
    }

    for key in ATTR_BOOL_KEYS:
        feats[f"attr_{re.sub(r'[^a-z0-9]+', '_', key.lower()).strip('_')}"] = 0
    for key in AMBIENCE_KEYS:
        feats[f"attr_ambience_{key}"] = 0
    for key in MEAL_KEYS:
        feats[f"attr_meal_{key}"] = 0
    for key in BEST_NIGHTS_KEYS:
        feats[f"attr_best_nights_{key}"] = 0
    for key in MUSIC_KEYS:
        feats[f"attr_music_{key}"] = 0

    if not d:
        return feats

    feats["attr_top_key_count"] = len(d)

    scalar_vals = []
    for k, v in d.items():
        if isinstance(v, dict):
            feats["attr_nested_dict_count"] += 1
        else:
            scalar_vals.append(v)

    for v in scalar_vals:
        if isinstance(v, bool):
            feats["attr_scalar_count"] += 1
            feats["attr_bool_true_count"] += int(v)
            feats["attr_bool_false_count"] += int(not v)
        elif isinstance(v, (int, float)):
            feats["attr_scalar_count"] += 1
            feats["attr_numeric_count"] += 1
        elif isinstance(v, str):
            feats["attr_scalar_count"] += 1
            feats["attr_string_count"] += 1

    for key in ATTR_BOOL_KEYS:
        val = _boolish(d.get(key))
        feats[f"attr_{re.sub(r'[^a-z0-9]+', '_', key.lower()).strip('_')}"] = int(bool(val)) if val is not None else 0

    pr = d.get("RestaurantsPriceRange2")
    try:
        feats["attr_price_range"] = int(str(pr).strip().strip("'\""))
        feats["attr_has_price_range"] = 1
    except Exception:
        feats["attr_price_range"] = 0
        feats["attr_has_price_range"] = 0

    wifi = _lower_text(d.get("WiFi"))
    if "free" in wifi:
        feats["attr_wifi_free"] = 1
    elif "paid" in wifi:
        feats["attr_wifi_paid"] = 1
    elif "no" in wifi:
        feats["attr_wifi_no"] = 1
    feats["attr_has_wifi"] = int(bool(wifi))

    alcohol = _lower_text(d.get("Alcohol"))
    feats["attr_alcohol_none"] = int("none" in alcohol)
    feats["attr_alcohol_beer_and_wine"] = int("beer" in alcohol or "wine" in alcohol)
    feats["attr_alcohol_full_bar"] = int(("full_bar" in alcohol) or ("full" in alcohol and "bar" in alcohol))

    noise = _lower_text(d.get("NoiseLevel"))
    feats["attr_noise_low"] = int(noise == "quiet" or noise == "low")
    feats["attr_noise_average"] = int("average" in noise)
    feats["attr_noise_loud"] = int("loud" in noise and "very" not in noise)
    feats["attr_noise_very_loud"] = int("very loud" in noise or "very_loud" in noise)

    attire = _lower_text(d.get("RestaurantsAttire"))
    feats["attr_attire_casual"] = int("casual" in attire)
    feats["attr_attire_dressy"] = int("dressy" in attire)
    feats["attr_attire_formal"] = int("formal" in attire)

    nested_families = {
        "BusinessParking": None,
        "Ambience": AMBIENCE_KEYS,
        "GoodForMeal": MEAL_KEYS,
        "BestNights": BEST_NIGHTS_KEYS,
        "Music": MUSIC_KEYS,
    }

    for family, subkeys in nested_families.items():
        node = d.get(family)
        if not isinstance(node, dict):
            continue
        feats["attr_family_count"] += 1

        if family == "BusinessParking":
            feats["attr_has_business_parking"] = 1
            true_count = 0
            for kk, vv in node.items():
                b = _boolish(vv)
                if b is True:
                    true_count += 1
                feats[f"attr_parking_{re.sub(r'[^a-z0-9]+', '_', str(kk).lower()).strip('_')}"] = int(bool(b)) if b is not None else 0
            feats["attr_parking_true_count"] = true_count
            feats["attr_parking_any"] = int(true_count > 0)

        elif family == "Ambience":
            feats["attr_has_ambience"] = 1
            true_count = 0
            for kk in AMBIENCE_KEYS:
                b = _boolish(node.get(kk))
                feats[f"attr_ambience_{kk}"] = int(bool(b)) if b is not None else 0
                true_count += int(b is True)
            feats["attr_ambience_true_count"] = true_count

        elif family == "GoodForMeal":
            feats["attr_has_good_for_meal"] = 1
            true_count = 0
            for kk in MEAL_KEYS:
                b = _boolish(node.get(kk))
                feats[f"attr_meal_{kk}"] = int(bool(b)) if b is not None else 0
                true_count += int(b is True)
            feats["attr_meal_true_count"] = true_count

        elif family == "BestNights":
            feats["attr_has_best_nights"] = 1
            true_count = 0
            for kk in BEST_NIGHTS_KEYS:
                b = _boolish(node.get(kk))
                feats[f"attr_best_nights_{kk}"] = int(bool(b)) if b is not None else 0
                true_count += int(b is True)
            feats["attr_best_nights_true_count"] = true_count

        elif family == "Music":
            feats["attr_has_music"] = 1
            true_count = 0
            for kk in MUSIC_KEYS:
                b = _boolish(node.get(kk))
                feats[f"attr_music_{kk}"] = int(bool(b)) if b is not None else 0
                true_count += int(b is True)
            feats["attr_music_true_count"] = true_count

    feats["attr_attribute_richness"] = (
        feats["attr_top_key_count"]
        + feats["attr_nested_dict_count"]
        + feats["attr_bool_true_count"]
        + feats["attr_parking_true_count"]
        + feats["attr_ambience_true_count"]
        + feats["attr_meal_true_count"]
        + feats["attr_best_nights_true_count"]
        + feats["attr_music_true_count"]
    )
    return feats


def _extract_categories_features(raw_cat):
    txt = "" if raw_cat is None else str(raw_cat)
    cats = [c.strip().lower() for c in txt.split(",") if c.strip()]
    joined = " | ".join(cats)

    feats = {
        "category_count": len(cats),
        "category_char_count_mean": float(np.mean([len(c) for c in cats])) if cats else 0.0,
        "category_char_count_std": float(np.std([len(c) for c in cats])) if len(cats) > 1 else 0.0,
        "category_char_count_min": float(np.min([len(c) for c in cats])) if cats else 0.0,
        "category_char_count_max": float(np.max([len(c) for c in cats])) if cats else 0.0,
        "category_word_count_mean": float(np.mean([len(c.split()) for c in cats])) if cats else 0.0,
        "category_word_count_std": float(np.std([len(c.split()) for c in cats])) if len(cats) > 1 else 0.0,
        "category_word_count_total": int(sum(len(c.split()) for c in cats)) if cats else 0,
        "category_broad_group_count": 0,
        "category_specificity": 0.0,
    }

    for group_name in CATEGORY_GROUPS:
        feats[group_name] = 0

    for group_name, kws in CATEGORY_GROUPS.items():
        hit = any(kw in joined for kw in kws)
        feats[group_name] = int(hit)
        feats["category_broad_group_count"] += int(hit)

    feats["category_specificity"] = feats["category_count"] / (feats["category_broad_group_count"] + 1e-6)
    return feats


def _parse_semistructured_batch(df: pl.DataFrame) -> pl.DataFrame:
    parsed_rows = []
    for h, a, c in zip(df["hours"].to_list(), df["attributes"].to_list(), df["categories"].to_list()):
        row_feats = {}
        row_feats.update(_extract_hours_features(h))
        row_feats.update(_extract_attributes_features(a))
        row_feats.update(_extract_categories_features(c))
        parsed_rows.append(row_feats)

    if len(parsed_rows) != df.height:
        raise ValueError(f"Parsing mismatch: {len(parsed_rows)} rows parsed for batch of height {df.height}")

    return pl.DataFrame(parsed_rows)


def build_business_features(
    business: pl.DataFrame,
    batch_size: int = 50_000,
):
    import polars as pl

    # =========================
    # 1. Limpieza inicial
    # =========================
    df = business.with_columns(
        pl.col("business_id"),
        pl.col("city").fill_null("unknown"),
        pl.col("state").fill_null("unknown"),
        pl.col("postal_code").fill_null("unknown"),
        pl.col("latitude").cast(pl.Float64),
        pl.col("longitude").cast(pl.Float64),
        pl.col("stars").cast(pl.Float64),
        pl.col("review_count").cast(pl.Float64),
        pl.col("is_open").fill_null(0).cast(pl.Int8),
        pl.col("attributes").fill_null(""),
        pl.col("categories").fill_null(""),
        pl.col("hours").fill_null(""),
    )

    # =========================
    # 2. Global stats
    # =========================
    global_stats = df.select(
        pl.col("stars").mean().alias("global_avg_stars"),
        pl.col("stars").std().alias("global_stars_std"),
    ).to_dicts()[0]

    # =========================
    # 3. Agregaciones geográficas
    # =========================
    city_stats = df.group_by("city").agg(
        pl.len().alias("city_business_count"),
        pl.col("stars").mean().alias("city_avg_stars"),
        pl.col("stars").std().alias("city_stars_std"),
    )

    state_stats = df.group_by("state").agg(
        pl.len().alias("state_business_count"),
        pl.col("stars").mean().alias("state_avg_stars"),
        pl.col("stars").std().alias("state_stars_std"),
    )

    df = (
        df.join(city_stats, on="city", how="left")
          .join(state_stats, on="state", how="left")
    )

    # =========================
    # 4. Features numéricas base
    # =========================
    df = df.with_columns(
        (pl.col("stars") - global_stats["global_avg_stars"]).alias("stars_vs_global"),
        (pl.col("stars") - pl.col("city_avg_stars")).alias("stars_vs_city"),
        (pl.col("stars") - pl.col("state_avg_stars")).alias("stars_vs_state"),

        (pl.col("review_count") + 1).log10().alias("review_count_log"),

        ((pl.col("stars") - pl.col("city_avg_stars")) /
         (pl.col("city_stars_std") + 1e-6)).alias("stars_city_z"),

        ((pl.col("stars") - pl.col("state_avg_stars")) /
         (pl.col("state_stars_std") + 1e-6)).alias("stars_state_z"),

        (pl.col("review_count") * pl.col("stars")).alias("popularity"),
    )

    # =========================
    # 5. Parsing pesado (batch)
    # =========================
    parsed_batches = []

    for start in range(0, df.height, batch_size):
        batch = df.slice(start, batch_size)

        rows = []
        for h, a, c in zip(
            batch["hours"].to_list(),
            batch["attributes"].to_list(),
            batch["categories"].to_list()
        ):
            # -------- HOURS --------
            open_days = 0
            total_minutes = 0

            try:
                h_dict = ast.literal_eval(h) if h else {}
            except:
                h_dict = {}

            for d in h_dict.values():
                try:
                    start_t, end_t = d.split("-")
                    sh, sm = map(int, start_t.split(":"))
                    eh, em = map(int, end_t.split(":"))
                    duration = (eh*60+em) - (sh*60+sm)
                    if duration > 0:
                        total_minutes += duration
                        open_days += 1
                except:
                    continue

            # -------- ATTR --------
            try:
                a_dict = ast.literal_eval(a) if a else {}
            except:
                a_dict = {}

            attr_count = len(a_dict)
            bool_count = sum(
                1 for v in a_dict.values()
                if str(v).lower() in ["true", "false"]
            )

            # -------- CAT --------
            cats = [x.strip().lower() for x in c.split(",") if x]

            rows.append({
                "open_days": open_days,
                "hours_total": total_minutes,
                "attr_count": attr_count,
                "attr_bool_count": bool_count,
                "cat_count": len(cats),
            })

        parsed_batches.append(pl.DataFrame(rows))

    parsed = pl.concat(parsed_batches)
    df = df.hstack(parsed)

    # =========================
    # 6. Encoding categóricas
    # =========================
    city_map = df.select("city").unique().with_row_count("city_idx")
    state_map = df.select("state").unique().with_row_count("state_idx")

    df = (
        df.join(city_map, on="city", how="left")
          .join(state_map, on="state", how="left")
    )

    # =========================
    # 7. Features finales
    # =========================
    df = df.with_columns(
        (pl.col("open_days") / 7).alias("open_ratio"),
        (pl.col("hours_total") / (pl.col("review_count") + 1)).alias("hours_per_review"),
        (pl.col("cat_count") / (pl.col("attr_count") + 1)).alias("cat_attr_ratio"),
    )

    # =========================
    # 8. Drop columnas no válidas
    # =========================
    df = df.drop([
        "city",
        "state",
        "postal_code",
        "attributes",
        "categories",
        "hours",
    ])

    return df, city_map, state_map

In [13]:
businesses, city_map, state_map = build_business_features(businesses, batch_size=50_000)
display(businesses.head())

/tmp/ipykernel_272111/197111573.py:723: DeprecationWarning: `DataFrame.with_row_count` is deprecated; use `with_row_index` instead. Note that the default column name has changed from 'row_nr' to 'index'.
  city_map = df.select("city").unique().with_row_count("city_idx")
/tmp/ipykernel_272111/197111573.py:724: DeprecationWarning: `DataFrame.with_row_count` is deprecated; use `with_row_index` instead. Note that the default column name has changed from 'row_nr' to 'index'.
  state_map = df.select("state").unique().with_row_count("state_idx")


business_id,name,address,latitude,longitude,stars,review_count,is_open,city_business_count,city_avg_stars,city_stars_std,state_business_count,state_avg_stars,state_stars_std,stars_vs_global,stars_vs_city,stars_vs_state,review_count_log,stars_city_z,stars_state_z,popularity,open_days,hours_total,attr_count,attr_bool_count,cat_count,city_idx,state_idx,open_ratio,hours_per_review,cat_attr_ratio
str,str,str,f64,f64,f64,f64,i8,u32,f64,f64,u32,f64,f64,f64,f64,f64,f64,f64,f64,f64,i64,i64,i64,i64,i64,u32,u32,f64,f64,f64
"""GDEEPQdYs2utMN-R4znZSA""","""Metro Self Storage - Largo""","""10501 S Belcher Rd""",27.868519,-82.743849,4.5,7.0,0,173,3.534682,1.019553,5226,3.58917,0.995679,0.910489,0.965318,0.91083,0.90309,0.946804,0.914782,31.5,6,2850,2,2,4,60,8,0.857143,356.25,1.333333
"""pbAq2NRG_2jCBI6fgRalvQ""","""Madewell""","""3301 Veterans Blvd, Ste 84""",30.005894,-90.15745,2.0,6.0,1,333,3.507508,0.915448,2044,3.678816,0.967255,-1.589511,-1.507508,-1.678816,0.845098,-1.64674,-1.735649,12.0,7,4320,1,1,5,557,1,1.0,617.142857,2.5
"""h4HP3Vc0dQq7SfSLal9qQw""","""Dollylocks""","""511 9th St N""",27.777785,-82.646673,4.5,7.0,0,334,3.628743,1.0932,5226,3.58917,0.995679,0.910489,0.871257,0.91083,0.90309,0.796978,0.914782,31.5,0,0,9,6,5,576,8,0.0,0.0,0.5
"""PndbFVbHE4730HDlghxv6g""","""Jim Browne Chrysler Jeep Dodge…","""10909 N Florida Ave""",28.048093,-82.458757,2.5,76.0,1,1779,3.543845,1.008842,5226,3.58917,0.995679,-1.089511,-1.043845,-1.08917,1.886491,-1.034695,-1.093895,190.0,7,4800,1,1,4,375,8,1.0,62.337662,2.0
"""IayDnngl0NooAbcoo62j-w""","""Emg Salons""","""324 S Falkenburg Rd""",27.948815,-82.334619,4.5,7.0,1,1779,3.543845,1.008842,5226,3.58917,0.995679,0.910489,0.956155,0.91083,0.90309,0.947774,0.914782,31.5,5,3300,9,5,6,375,8,0.714286,412.5,0.6


## 3. Creación de los datos de entrada - Online

In [14]:
def build_review_features(
    reviews: pl.DataFrame,
    reference_date=None,
    keep_keys: bool = True,
) -> pl.DataFrame:

    import polars as pl
    import math

    def _safe_log10(expr):
        return (
            expr.cast(pl.Float64)
            .fill_null(0)
            .clip(lower_bound=0)
            .add(1)
            .log10()
            .fill_null(0)
        )

    # =========================
    # 1. Base
    # =========================
    df = reviews.with_columns(
        pl.col("datetime").cast(pl.Datetime).alias("review_dt"),
        pl.col("date").cast(pl.Date).alias("review_date"),
        pl.col("useful").fill_null(0),
        pl.col("funny").fill_null(0),
        pl.col("cool").fill_null(0),
    )

    # =========================
    # 2. Features temporales
    # =========================
    df = df.with_columns(
        pl.col("review_dt").dt.year().alias("review_year"),
        pl.col("review_dt").dt.month().alias("review_month"),
        pl.col("review_dt").dt.weekday().alias("review_weekday"),
        pl.col("review_dt").dt.hour().alias("review_hour"),
        pl.col("review_dt").dt.minute().alias("review_minute"),
        pl.col("review_dt").dt.ordinal_day().alias("review_day_of_year"),
    )

    df = df.with_columns(
        (pl.col("review_hour") + pl.col("review_minute") / 60.0).alias("review_hour_float"),
        (pl.col("review_weekday") >= 6).cast(pl.Int8).alias("review_is_weekend"),
        (pl.col("review_weekday") <= 5).cast(pl.Int8).alias("review_is_workday"),
    )

    # =========================
    # 3. Cyclical encoding
    # =========================
    df = df.with_columns(
        (2 * math.pi * pl.col("review_hour_float") / 24).sin().alias("hour_sin"),
        (2 * math.pi * pl.col("review_hour_float") / 24).cos().alias("hour_cos"),
        (2 * math.pi * (pl.col("review_weekday") / 7)).sin().alias("weekday_sin"),
        (2 * math.pi * (pl.col("review_weekday") / 7)).cos().alias("weekday_cos"),
    )

    # =========================
    # 4. Contexto comida
    # =========================
    df = df.with_columns(
        pl.col("review_hour").is_between(6, 9).cast(pl.Int8).alias("is_breakfast"),
        pl.col("review_hour").is_between(11, 14).cast(pl.Int8).alias("is_lunch"),
        pl.col("review_hour").is_between(17, 21).cast(pl.Int8).alias("is_dinner"),
        ((pl.col("review_hour") >= 22) | (pl.col("review_hour") <= 5))
        .cast(pl.Int8)
        .alias("is_late_night"),
    )

    # =========================
    # 5. Feedback (SEPARADO!)
    # =========================
    df = df.with_columns(
        (pl.col("useful") + pl.col("funny") + pl.col("cool")).alias("feedback_total"),
    )

    df = df.with_columns(
        _safe_log10(pl.col("useful")).alias("useful_log"),
        _safe_log10(pl.col("funny")).alias("funny_log"),
        _safe_log10(pl.col("cool")).alias("cool_log"),
        _safe_log10(pl.col("feedback_total")).alias("feedback_total_log"),

        (pl.col("useful") / (pl.col("feedback_total") + 1)).alias("useful_ratio"),
        (pl.col("funny") / (pl.col("feedback_total") + 1)).alias("funny_ratio"),
        (pl.col("cool") / (pl.col("feedback_total") + 1)).alias("cool_ratio"),
    )

    # =========================
    # 6. Recencia
    # =========================
    if reference_date is not None:
        df = df.with_columns(
            (
                pl.lit(reference_date).cast(pl.Date)
                - pl.col("review_date")
            ).dt.total_days().alias("days_since")
        )

        df = df.with_columns(
            _safe_log10(pl.col("days_since")).alias("days_since_log")
        )

    # =========================
    # 7. Interacciones (CORREGIDO)
    # =========================
    df = df.with_columns(
        (pl.col("review_is_weekend") * pl.col("is_late_night")).alias("weekend_late"),
        (pl.col("review_is_workday") * pl.col("is_dinner")).alias("workday_dinner"),
        (pl.col("review_is_weekend") * pl.col("is_dinner")).alias("weekend_dinner"),
    )

    # =========================
    # 8. Limpieza
    # =========================
    drop_cols = ["review_dt", "review_date", "review_id"]

    if not keep_keys:
        drop_cols += ["user_id", "business_id"]

    df = df.drop([c for c in drop_cols if c in df.columns])

    return df

In [15]:
train = build_review_features(train, reference_date=date(2023, 1, 1), keep_keys=True)
test = build_review_features(test, reference_date=date(2023, 1, 1), keep_keys=True)

In [16]:
display(train.head())

user_id,business_id,stars,useful,funny,cool,date,datetime,review_year,review_month,review_weekday,review_hour,review_minute,review_day_of_year,review_hour_float,review_is_weekend,review_is_workday,hour_sin,hour_cos,weekday_sin,weekday_cos,is_breakfast,is_lunch,is_dinner,is_late_night,feedback_total,useful_log,funny_log,cool_log,feedback_total_log,useful_ratio,funny_ratio,cool_ratio,days_since,days_since_log,weekend_late,workday_dinner,weekend_dinner
str,str,f64,i64,i64,i64,date,datetime[μs],i32,i8,i8,i8,i8,i16,f64,i8,i8,f64,f64,f64,f64,i8,i8,i8,i8,i64,f64,f64,f64,f64,f64,f64,f64,i64,f64,i8,i8,i8
"""-1BSu2dt_rOAqllw9ZDXtA""","""smkZq4G1AOm4V6p3id5sww""",5.0,0,0,0,2016-09-30,2016-09-30 15:49:32,2016,9,5,15,49,274,15.816667,0,1,-0.841039,-0.540974,-0.974928,-0.222521,0,0,0,0,0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,2284,3.358886,0,0,0
"""xpe178ng_gj5X6HgqtOing""","""96_c_7twb7hYRZ9HHrq01g""",1.0,2,0,1,2020-12-09,2020-12-09 14:39:51,2020,12,3,14,39,344,14.65,0,1,-0.639439,-0.768842,0.433884,-0.900969,0,1,0,0,3,0.477121,0.0,0.30103,0.60206,0.5,0.0,0.25,753,2.877371,0,0,0
"""axbaerf2Fk92OB4b9_peVA""","""e0AYjKfSF0DL-5C1CpOq6Q""",4.0,0,0,0,2013-09-04,2013-09-04 16:19:51,2013,9,3,16,19,247,16.316667,0,1,-0.904455,-0.426569,0.433884,-0.900969,0,0,0,0,0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,3406,3.532372,0,0,0
"""_GOiybb0rImYKJfwyxEaGg""","""vF-uptiQ34pVLHJKzPHUlA""",5.0,0,0,0,2019-03-02,2019-03-02 12:24:14,2019,3,6,12,24,61,12.4,1,0,-0.104528,-0.994522,-0.781831,0.62349,0,1,0,0,0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1401,3.146748,0,0,0
"""ojWKg3B5pH3ncAsxun3kUw""","""X28XK71RuEXPapeyUOwNzg""",5.0,10,4,7,2020-04-23,2020-04-23 18:26:29,2020,4,4,18,26,114,18.433333,0,1,-0.993572,0.113203,-0.433884,-0.900969,0,0,1,0,21,1.041393,0.69897,0.90309,1.342423,0.454545,0.181818,0.318182,983,2.992995,0,1,0


In [19]:
businesses.head()

business_id,name,address,latitude,longitude,stars,review_count,is_open,city_business_count,city_avg_stars,city_stars_std,state_business_count,state_avg_stars,state_stars_std,stars_vs_global,stars_vs_city,stars_vs_state,review_count_log,stars_city_z,stars_state_z,popularity,open_days,hours_total,attr_count,attr_bool_count,cat_count,city_idx,state_idx,open_ratio,hours_per_review,cat_attr_ratio
str,str,str,f64,f64,f64,f64,i8,u32,f64,f64,u32,f64,f64,f64,f64,f64,f64,f64,f64,f64,i64,i64,i64,i64,i64,u32,u32,f64,f64,f64
"""GDEEPQdYs2utMN-R4znZSA""","""Metro Self Storage - Largo""","""10501 S Belcher Rd""",27.868519,-82.743849,4.5,7.0,0,173,3.534682,1.019553,5226,3.58917,0.995679,0.910489,0.965318,0.91083,0.90309,0.946804,0.914782,31.5,6,2850,2,2,4,60,8,0.857143,356.25,1.333333
"""pbAq2NRG_2jCBI6fgRalvQ""","""Madewell""","""3301 Veterans Blvd, Ste 84""",30.005894,-90.15745,2.0,6.0,1,333,3.507508,0.915448,2044,3.678816,0.967255,-1.589511,-1.507508,-1.678816,0.845098,-1.64674,-1.735649,12.0,7,4320,1,1,5,557,1,1.0,617.142857,2.5
"""h4HP3Vc0dQq7SfSLal9qQw""","""Dollylocks""","""511 9th St N""",27.777785,-82.646673,4.5,7.0,0,334,3.628743,1.0932,5226,3.58917,0.995679,0.910489,0.871257,0.91083,0.90309,0.796978,0.914782,31.5,0,0,9,6,5,576,8,0.0,0.0,0.5
"""PndbFVbHE4730HDlghxv6g""","""Jim Browne Chrysler Jeep Dodge…","""10909 N Florida Ave""",28.048093,-82.458757,2.5,76.0,1,1779,3.543845,1.008842,5226,3.58917,0.995679,-1.089511,-1.043845,-1.08917,1.886491,-1.034695,-1.093895,190.0,7,4800,1,1,4,375,8,1.0,62.337662,2.0
"""IayDnngl0NooAbcoo62j-w""","""Emg Salons""","""324 S Falkenburg Rd""",27.948815,-82.334619,4.5,7.0,1,1779,3.543845,1.008842,5226,3.58917,0.995679,0.910489,0.956155,0.91083,0.90309,0.947774,0.914782,31.5,5,3300,9,5,6,375,8,0.714286,412.5,0.6


In [17]:
train.write_parquet("train_processed.parquet")
test.write_parquet("test_processed.parquet")
users.write_parquet("users_processed.parquet")
businesses.write_parquet("businesses_processed.parquet")

# Modelo

In [ ]:
# ============================================================
# 0) Imports
# ============================================================
from __future__ import annotations

import math
import copy
from dataclasses import dataclass
from typing import Dict, List, Optional, Tuple, Any

import numpy as np
import pandas as pd

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

from sklearn.model_selection import train_test_split
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler


# ============================================================
# 1) Helpers de conversión / merge
# ============================================================
def to_pandas(df: Any) -> pd.DataFrame:
    """
    Convierte Polars -> Pandas si hace falta.
    Si ya es pandas, devuelve copia.
    """
    try:
        import polars as pl
        if isinstance(df, pl.DataFrame):
            return df.to_pandas()
    except Exception:
        pass

    if isinstance(df, pd.DataFrame):
        return df.copy()

    raise TypeError(f"Tipo no soportado: {type(df)}")


def prepare_merged_df(
    train: Any,
    users: Any,
    businesses: Any,
    user_key_col: str = "user_id",
    business_key_col: str = "business_id",
    target_col: str = "stars",
) -> pd.DataFrame:
    """
    Merge:
      train + users + businesses

    Se prefijan columnas para evitar colisiones:
      user_*
      biz_*

    business_key_col debe existir en businesses.
    """
    train = to_pandas(train)
    users = to_pandas(users)
    businesses = to_pandas(businesses)

    if user_key_col not in train.columns:
        raise ValueError(f"'{user_key_col}' no existe en train")
    if user_key_col not in users.columns:
        raise ValueError(f"'{user_key_col}' no existe en users")
    if business_key_col not in train.columns:
        raise ValueError(f"'{business_key_col}' no existe en train")
    if business_key_col not in businesses.columns:
        raise ValueError(
            f"'{business_key_col}' no existe en businesses. "
            f"Revisa el nombre de la clave."
        )

    # Prefijos para evitar colisiones
    users_renamed = users.rename(
        columns={c: f"user_{c}" for c in users.columns if c != user_key_col}
    )
    businesses_renamed = businesses.rename(
        columns={c: f"biz_{c}" for c in businesses.columns if c != business_key_col}
    )

    df = train.merge(users_renamed, on=user_key_col, how="left")
    df = df.merge(businesses_renamed, left_on=business_key_col, right_on=business_key_col, how="left")

    # Quitar columnas crudas no numéricas si existen
    for c in ["date", "datetime"]:
        if c in df.columns:
            df = df.drop(columns=[c])

    return df


# ============================================================
# 2) Normalización / selección de columnas
# ============================================================
COUNT_LIKE_EXACT = {
    # review
    "feedback_total",
    "days_since",
    # user
    "user_review_count",
    "user_useful",
    "user_funny",
    "user_cool",
    "user_fans",
    "user_friends_count",
    "user_elite_count",
    "user_account_age_days",
    "user_total_compliments",
    "user_compl_total_log",  # por si existe un nombre raro, no hace daño
    "user_friends_edge_count",
    "user_friends_unique_count",
    "user_friends_valid_profile_count",
    "user_reviews_per_year",
    "user_fans_per_year",
    "user_friends_per_year",
    "user_elite_per_year",
    "user_reviews_per_day",
    "user_fans_per_day",
    # business
    "biz_review_count",
    "biz_city_business_count",
    "biz_state_business_count",
    "biz_open_days",
    "biz_hours_total",
    "biz_attr_count",
    "biz_attr_bool_count",
    "biz_cat_count",
    "biz_popularity",
}

COUNT_LIKE_CONTAINS = (
    "_count",
    "_counts",
    "review_count",
    "fans_per_",
    "friends_per_",
    "elite_per_",
    "reviews_per_day",
    "reviews_per_year",
    "fans_per_day",
    "fans_per_year",
    "compliments_per_",
    "total_compliments",
    "account_age_days",
    "days_since",
    "open_days",
    "hours_total",
    "hours_per_review",
    "feedback_total",
    "popularity",
    "friend_count",
)

NON_LOG_PREFIXES = ("is_", "has_")
NON_LOG_SUFFIXES = ("_log", "_sin", "_cos", "_ratio", "_z", "_std", "_cv", "_sq")

EXCLUDE_FROM_NUMERIC = {
    "stars",  # target
}

# Variables temporales ya codificadas cíclicamente o discretamente
TEMPORAL_EXACT_EXCLUDE = {
    "review_year",
    "review_month",
    "review_weekday",
    "review_hour",
    "review_minute",
    "review_day_of_year",
    "review_hour_float",
    "review_is_weekend",
    "review_is_workday",
    "signup_year",
    "signup_month",
    "signup_quarter",
    "signup_weekday",
    "signup_day_of_year",
    "signup_day_of_month",
    "signup_week_of_year",
    "signup_hour",
    "signup_is_weekend",
    "signup_is_workday",
    "signup_is_month_start",
    "signup_is_month_end",
    "signup_season_id",
    "account_age_bucket_id",
    "city_idx",
    "state_idx",
}

TIME_RAW_EXCLUDE = {
    "review_year",
    "review_month",
    "review_weekday",
    "review_hour",
    "review_minute",
    "review_day_of_year",
    "review_hour_float",
}

USER_TIME_RAW_EXCLUDE = {
    "signup_month",
    "signup_weekday",
    "signup_day_of_year",
    "signup_hour"
}

def is_numeric_series(s: pd.Series) -> bool:
    return pd.api.types.is_numeric_dtype(s)

def should_log1p(col: str) -> bool:
    c = col.lower()

    if c in EXCLUDE_FROM_NUMERIC:
        return False
    if c in TEMPORAL_EXACT_EXCLUDE:
        return False
    if c.endswith(NON_LOG_SUFFIXES):
        return False
    if c.startswith(NON_LOG_PREFIXES):
        return False
    if c in COUNT_LIKE_EXACT:
        return True
    if any(tok in c for tok in COUNT_LIKE_CONTAINS):
        # evita tocar ratios / logs / z-scores / sin-cos
        return True

    return False


# ============================================================
# 3) Preprocesador tabular
# ============================================================
@dataclass
class GroupSpec:
    cols: List[str]
    log1p_cols: List[str]
    standardize_cols: List[str]


class TabularPreprocessor:
    """
    - Encoders para user_id y business_id
    - Imputer + StandardScaler para:
        review features
        user features
        business features
    - log1p en contadores / variables fuertemente sesgadas
    """

    def __init__(
        self,
        user_key_col: str = "user_id",
        business_key_col: str = "business_id",
        target_col: str = "stars",
    ):
        self.user_key_col = user_key_col
        self.business_key_col = business_key_col
        self.target_col = target_col

        self.review_spec: Optional[GroupSpec] = None
        self.user_spec: Optional[GroupSpec] = None
        self.biz_spec: Optional[GroupSpec] = None

        self.review_imputer: Optional[SimpleImputer] = None
        self.user_imputer: Optional[SimpleImputer] = None
        self.biz_imputer: Optional[SimpleImputer] = None

        self.review_scaler: Optional[StandardScaler] = None
        self.user_scaler: Optional[StandardScaler] = None
        self.biz_scaler: Optional[StandardScaler] = None

        self.user2idx: Dict[str, int] = {}
        self.biz2idx: Dict[str, int] = {}

    @staticmethod
    def _safe_str(v: Any) -> str:
        if pd.isna(v):
            return "__NA__"
        return str(v)

    def _build_group_spec(self, df: pd.DataFrame, cols: List[str]) -> GroupSpec:
        numeric_cols = [c for c in cols if is_numeric_series(df[c])]
        log1p_cols = [c for c in numeric_cols if should_log1p(c)]
        std_cols = numeric_cols[:]  # todos los numéricos se estandarizan tras transformar
        return GroupSpec(cols=numeric_cols, log1p_cols=log1p_cols, standardize_cols=std_cols)

    def fit(self, df: pd.DataFrame) -> "TabularPreprocessor":
        df = df.copy()

        # IDs
        users = df[self.user_key_col].map(self._safe_str).fillna("__NA__").unique().tolist()
        bizs = df[self.business_key_col].map(self._safe_str).fillna("__NA__").unique().tolist()

        self.user2idx = {"__UNK__": 0}
        self.user2idx.update({u: i + 1 for i, u in enumerate(users)})

        self.biz2idx = {"__UNK__": 0}
        self.biz2idx.update({b: i + 1 for i, b in enumerate(bizs)})

        # Columnas por grupo
        review_cols = []
        user_cols = []
        biz_cols = []

        for c in df.columns:
            if c in {self.user_key_col, self.business_key_col, self.target_col, "date", "datetime"} or c in TIME_RAW_EXCLUDE or c in USER_TIME_RAW_EXCLUDE:
                continue
            if not is_numeric_series(df[c]):
                continue
            if c.startswith("user_"):
                user_cols.append(c)
            elif c.startswith("biz_"):
                biz_cols.append(c)
            else:
                review_cols.append(c)

        self.review_spec = self._build_group_spec(df, review_cols)
        self.user_spec = self._build_group_spec(df, user_cols)
        self.biz_spec = self._build_group_spec(df, biz_cols)

        # Fit imputers + scalers por grupo
        self.review_imputer = SimpleImputer(strategy="median")
        self.user_imputer = SimpleImputer(strategy="median")
        self.biz_imputer = SimpleImputer(strategy="median")

        self.review_scaler = StandardScaler()
        self.user_scaler = StandardScaler()
        self.biz_scaler = StandardScaler()

        self._fit_group(df, self.review_spec, self.review_imputer, self.review_scaler)
        self._fit_group(df, self.user_spec, self.user_imputer, self.user_scaler)
        self._fit_group(df, self.biz_spec, self.biz_imputer, self.biz_scaler)

        return self

    def _fit_group(
        self,
        df: pd.DataFrame,
        spec: GroupSpec,
        imputer: SimpleImputer,
        scaler: StandardScaler,
    ) -> None:
        if len(spec.cols) == 0:
            # grupo vacío
            imputer.fit(np.zeros((1, 1)))
            scaler.fit(np.zeros((1, 1)))
            return

        X = self._transform_raw_numeric(df, spec.cols, spec.log1p_cols)
        X = imputer.fit_transform(X)
        scaler.fit(X)

    @staticmethod
    def _transform_raw_numeric(
        df: pd.DataFrame,
        cols: List[str],
        log1p_cols: List[str],
    ) -> np.ndarray:
        if len(cols) == 0:
            return np.zeros((len(df), 1), dtype=np.float32)

        out = []
        for c in cols:
            x = df[c].astype("float32").to_numpy(copy=True)

            # Imputación temporal por mediana se hará luego
            if c in log1p_cols:
                x = np.clip(x, a_min=0, a_max=None)
                x = np.log1p(x)

            out.append(x)

        X = np.column_stack(out).astype(np.float32)
        return X

    def transform(self, df: pd.DataFrame, include_target: bool = True) -> Dict[str, np.ndarray]:
        if self.review_spec is None or self.user_spec is None or self.biz_spec is None:
            raise RuntimeError("Primero llama a fit().")

        df = df.copy()

        # IDs
        user_ids = df[self.user_key_col].map(self._safe_str).fillna("__NA__").map(lambda x: self.user2idx.get(x, 0)).to_numpy()
        biz_ids = df[self.business_key_col].map(self._safe_str).fillna("__NA__").map(lambda x: self.biz2idx.get(x, 0)).to_numpy()

        review_x = self._transform_group(df, self.review_spec, self.review_imputer, self.review_scaler)
        user_x = self._transform_group(df, self.user_spec, self.user_imputer, self.user_scaler)
        biz_x = self._transform_group(df, self.biz_spec, self.biz_imputer, self.biz_scaler)

        out = {
            "user_id": user_ids.astype(np.int64),
            "business_id": biz_ids.astype(np.int64),
            "review_x": review_x.astype(np.float32),
            "user_x": user_x.astype(np.float32),
            "biz_x": biz_x.astype(np.float32),
        }

        if include_target:
            if self.target_col not in df.columns:
                raise ValueError(f"'{self.target_col}' no existe en df")
            y = df[self.target_col].astype(int).to_numpy()  # 1..5
            out["y"] = y.astype(np.int64)

        return out

    def _transform_group(
        self,
        df: pd.DataFrame,
        spec: GroupSpec,
        imputer: SimpleImputer,
        scaler: StandardScaler,
    ) -> np.ndarray:
        if len(spec.cols) == 0:
            return np.zeros((len(df), 1), dtype=np.float32)

        X = self._transform_raw_numeric(df, spec.cols, spec.log1p_cols)
        X = imputer.transform(X)
        X = scaler.transform(X)
        return X


# ============================================================
# 4) Dataset
# ============================================================
class ReviewDataset(Dataset):
    def __init__(self, data: Dict[str, np.ndarray], has_target: bool = True):
        self.user_id = torch.tensor(data["user_id"], dtype=torch.long)
        self.business_id = torch.tensor(data["business_id"], dtype=torch.long)
        self.review_x = torch.tensor(data["review_x"], dtype=torch.float32)
        self.user_x = torch.tensor(data["user_x"], dtype=torch.float32)
        self.biz_x = torch.tensor(data["biz_x"], dtype=torch.float32)
        self.has_target = has_target
        if has_target:
            self.y = torch.tensor(data["y"], dtype=torch.long) - 1  # 0..4
        else:
            self.y = None

    def __len__(self) -> int:
        return self.user_id.shape[0]

    def __getitem__(self, idx: int):
        item = {
            "user_id": self.user_id[idx],
            "business_id": self.business_id[idx],
            "review_x": self.review_x[idx],
            "user_x": self.user_x[idx],
            "biz_x": self.biz_x[idx],
        }
        if self.has_target:
            item["y"] = self.y[idx]
        return item


# ============================================================
# 5) Modelo
# ============================================================
class FeatureGate(nn.Module):
    """
    Gating simple: aprende un peso por feature.
    x -> sigmoid(Wx+b) -> x * gate
    """
    def __init__(self, in_dim: int):
        super().__init__()
        self.gate = nn.Linear(in_dim, in_dim)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        g = torch.sigmoid(self.gate(x))
        return x * g


class AttentionTower(nn.Module):
    """
    Torre tabular con autoatención sobre tokens latentes.
    Convierte un vector plano en una pequeña secuencia de tokens,
    aplica self-attention y luego agrega con pooling.
    """
    def __init__(
        self,
        in_dim: int,
        out_dim: int,
        num_tokens: int = 8,
        token_dim: int = 32,
        num_heads: int = 4,
        ff_dim: int = 128,
        dropout: float = 0.15,
    ):
        super().__init__()
        assert token_dim % num_heads == 0, "token_dim debe ser divisible por num_heads"

        self.in_norm = nn.LayerNorm(in_dim)
        self.gate = FeatureGate(in_dim)

        self.num_tokens = num_tokens
        self.token_dim = token_dim

        # Proyección del vector tabular a una secuencia de tokens latentes
        self.proj = nn.Linear(in_dim, num_tokens * token_dim)
        self.pos_emb = nn.Parameter(torch.randn(1, num_tokens, token_dim) * 0.02)

        self.attn = nn.MultiheadAttention(
            embed_dim=token_dim,
            num_heads=num_heads,
            dropout=dropout,
            batch_first=True,
        )
        self.attn_norm = nn.LayerNorm(token_dim)

        self.ffn = nn.Sequential(
            nn.Linear(token_dim, ff_dim),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(ff_dim, token_dim),
        )
        self.ffn_norm = nn.LayerNorm(token_dim)

        self.out = nn.Sequential(
            nn.Linear(token_dim, out_dim),
            nn.LayerNorm(out_dim),
            nn.ReLU(),
            nn.Dropout(dropout),
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x = self.in_norm(x)
        x = self.gate(x)

        B = x.size(0)

        # [B, in_dim] -> [B, num_tokens, token_dim]
        t = self.proj(x).view(B, self.num_tokens, self.token_dim)
        t = t + self.pos_emb

        attn_out, _ = self.attn(t, t, t, need_weights=False)
        t = self.attn_norm(t + attn_out)

        ffn_out = self.ffn(t)
        t = self.ffn_norm(t + ffn_out)

        # pooling sobre tokens
        pooled = t.mean(dim=1)
        return self.out(pooled)

class CornHead(nn.Module):
    """
    CORN para 5 clases => 4 logits.
    """
    def __init__(self, in_dim: int, num_classes: int = 5):
        super().__init__()
        assert num_classes >= 3
        self.num_classes = num_classes
        self.fc = nn.Linear(in_dim, num_classes - 1)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.fc(x)


class ReviewRatingNet(nn.Module):
    def __init__(
        self,
        num_users: int,
        num_businesses: int,
        review_dim: int,
        user_dim: int,
        biz_dim: int,
        user_emb_dim: int = 32,
        biz_emb_dim: int = 32,
        tower_out_dim: int = 64,
        fusion_hidden_dims: Tuple[int, ...] = (256, 128),
        dropout: float = 0.20,
        num_classes: int = 5,
    ):
        super().__init__()
        self.num_classes = num_classes

        self.user_emb = nn.Embedding(num_users, user_emb_dim)
        self.biz_emb = nn.Embedding(num_businesses, biz_emb_dim)

        self.user_tower = AttentionTower(
            user_emb_dim + user_dim,
            tower_out_dim,
            num_tokens=8,
            token_dim=32,
            num_heads=4,
            ff_dim=128,
            dropout=dropout,
        )

        self.biz_tower = AttentionTower(
            biz_emb_dim + biz_dim,
            tower_out_dim,
            num_tokens=8,
            token_dim=32,
            num_heads=4,
            ff_dim=128,
            dropout=dropout,
        )

        self.review_tower = AttentionTower(
            review_dim,
            tower_out_dim // 2,
            num_tokens=6,
            token_dim=24,
            num_heads=4,
            ff_dim=96,
            dropout=dropout,
        )
        # Interacciones:
        # [u, b, r, u*b, |u-b|, u+b, u*r, b*r]
        fusion_in_dim = (
            tower_out_dim
            + tower_out_dim
            + (tower_out_dim // 2)
            + tower_out_dim
            + tower_out_dim
            + tower_out_dim
            + tower_out_dim
            + tower_out_dim
        )

        fusion_layers = []
        dims = (fusion_in_dim,) + tuple(fusion_hidden_dims)
        for i in range(len(dims) - 1):
            fusion_layers.append(nn.Linear(dims[i], dims[i + 1]))
            fusion_layers.append(nn.LayerNorm(dims[i + 1]))
            fusion_layers.append(nn.ReLU())
            fusion_layers.append(nn.Dropout(dropout))

        self.fusion = nn.Sequential(*fusion_layers)
        self.head = CornHead(fusion_hidden_dims[-1], num_classes=num_classes)

    def forward(
        self,
        user_id: torch.Tensor,
        business_id: torch.Tensor,
        review_x: torch.Tensor,
        user_x: torch.Tensor,
        biz_x: torch.Tensor,
    ) -> torch.Tensor:
        ue = self.user_emb(user_id)
        be = self.biz_emb(business_id)

        u = self.user_tower(torch.cat([ue, user_x], dim=1))
        b = self.biz_tower(torch.cat([be, biz_x], dim=1))
        r = self.review_tower(review_x)

        # Reutilizamos r en interacciones con u/b
        r_u = F.pad(r, (0, u.shape[1] - r.shape[1])) if r.shape[1] < u.shape[1] else r[:, :u.shape[1]]
        r_b = r_u

        inter_ub = u * b
        abs_ub = torch.abs(u - b)
        sum_ub = u + b
        inter_ur = u * r_u
        inter_br = b * r_b

        z = torch.cat([u, b, r, inter_ub, abs_ub, sum_ub, inter_ur, inter_br], dim=1)
        z = self.fusion(z)
        logits = self.head(z)
        return logits


# ============================================================
# 6) CORN loss / predicción
# ============================================================
def corn_targets(y: torch.Tensor, num_classes: int = 5) -> torch.Tensor:
    """
    y: tensor [B] con etiquetas 0..4
    return: [B, 4] con targets ordinales
    """
    thresholds = torch.arange(num_classes - 1, device=y.device).unsqueeze(0)  # [1, 4]
    return (y.unsqueeze(1) > thresholds).float()


@torch.no_grad()
def corn_predict(logits: torch.Tensor) -> torch.Tensor:
    """
    Devuelve predicción 1..5
    """
    probs = torch.sigmoid(logits)
    pred = 1 + (probs > 0.5).sum(dim=1)
    return pred.clamp(1, 5)

def compute_corn_pos_weight(y: np.ndarray, num_classes: int = 5) -> torch.Tensor:
    """
    y debe venir en 0..4
    Devuelve peso por umbral: [w1, w2, w3, w4]
    """
    weights = []
    n = len(y)
    for t in range(num_classes - 1):
        pos = np.sum(y > t)
        neg = n - pos
        w = neg / max(pos, 1)
        weights.append(w)
    return torch.tensor(weights, dtype=torch.float32)

def corn_loss(logits: torch.Tensor, y: torch.Tensor, pos_weight: torch.Tensor) -> torch.Tensor:
    target = corn_targets(y, num_classes=5)
    pw = pos_weight.to(logits.device)
    loss_fn = nn.BCEWithLogitsLoss(pos_weight=pw, reduction="none")
    loss = loss_fn(logits, target)
    return loss.mean()


# ============================================================
# 7) Train / eval
# ============================================================
def batch_to_device(batch: Dict[str, torch.Tensor], device: torch.device) -> Dict[str, torch.Tensor]:
    return {k: v.to(device) for k, v in batch.items()}

@torch.no_grad()
def evaluate(model: nn.Module, loader: DataLoader, device: torch.device, pw:List[float]) -> Dict[str, float]:
    model.eval()
    losses = []
    all_true = []
    all_pred = []

    for batch in loader:
        batch = batch_to_device(batch, device)
        logits = model(
            batch["user_id"],
            batch["business_id"],
            batch["review_x"],
            batch["user_x"],
            batch["biz_x"],
        )
        loss = corn_loss(logits, batch["y"], pw)
        pred = corn_predict(logits)

        losses.append(loss.item())
        all_true.append((batch["y"] + 1).cpu().numpy())  # back to 1..5
        all_pred.append(pred.cpu().numpy())

    y_true = np.concatenate(all_true)
    y_pred = np.concatenate(all_pred)

    mae = np.mean(np.abs(y_true - y_pred))
    acc = np.mean(y_true == y_pred)
    within_1 = np.mean(np.abs(y_true - y_pred) <= 1)

    return {
        "loss": float(np.mean(losses)),
        "mae": float(mae),
        "acc": float(acc),
        "within_1": float(within_1),
    }


def train_model(
    model: nn.Module,
    train_loader: DataLoader,
    val_loader: DataLoader,
    device: torch.device,
    pos_weight: List[float],
    epochs: int = 20,
    lr: float = 3e-4,
    weight_decay: float = 1e-4,
    patience: int = 5,
) -> nn.Module:
    optimizer = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=weight_decay)
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimizer, mode="min", patience=2, factor=0.5
    )

    best_state = None
    best_val = float("inf")
    no_improve = 0

    model.to(device)
    print("A entrenar!")
    for epoch in range(1, epochs + 1):
        model.train()
        train_losses = []

        for batch in train_loader:
            batch = batch_to_device(batch, device)

            optimizer.zero_grad(set_to_none=True)
            logits = model(
                batch["user_id"],
                batch["business_id"],
                batch["review_x"],
                batch["user_x"],
                batch["biz_x"],
            )
            loss = corn_loss(logits, batch["y"], pos_weight=pos_weight)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=2.0)
            optimizer.step()

            train_losses.append(loss.item())

        train_loss = float(np.mean(train_losses))
        val_metrics = evaluate(model, val_loader, device, pos_weight)
        scheduler.step(val_metrics["loss"])

        print(
            f"Epoch {epoch:02d} | "
            f"train_loss={train_loss:.4f} | "
            f"val_loss={val_metrics['loss']:.4f} | "
            f"val_mae={val_metrics['mae']:.4f} | "
            f"val_acc={val_metrics['acc']:.4f} | "
            f"val_within_1={val_metrics['within_1']:.4f}"
        )

        if val_metrics["loss"] < best_val:
            best_val = val_metrics["loss"]
            best_state = copy.deepcopy(model.state_dict())
            no_improve = 0
        else:
            no_improve += 1
            if no_improve >= patience:
                print("Early stopping.")
                break

    if best_state is not None:
        model.load_state_dict(best_state)

    return model


# ============================================================
# 8) Pipeline completo de ejemplo
# ============================================================
def build_and_train(
    train: Any,
    users: Any,
    businesses: Any,
    user_key_col: str = "user_id",
    business_key_col: str = "business_id",
    target_col: str = "stars",
    batch_size: int = 256,
    epochs: int = 20,
    lr: float = 3e-4,
    weight_decay: float = 1e-4,
    val_size: float = 0.15,
    random_state: int = 42,
) -> Tuple[ReviewRatingNet, TabularPreprocessor]:
    """
    Devuelve modelo entrenado + preprocesador.
    """
    df = prepare_merged_df(
        train=train,
        users=users,
        businesses=businesses,
        user_key_col=user_key_col,
        business_key_col=business_key_col,
        target_col=target_col,
    )

    # Split estratificado por stars
    idx = np.arange(len(df))
    y = df[target_col].astype(int).to_numpy()

    train_idx, val_idx = train_test_split(
        idx,
        test_size=val_size,
        random_state=random_state,
        stratify=y,
    )

    df_train = df.iloc[train_idx].reset_index(drop=True)
    df_val = df.iloc[val_idx].reset_index(drop=True)
    y_train = df_train[target_col].astype(int).to_numpy() - 1
    pos_weight = compute_corn_pos_weight(y_train, num_classes=5)

    # Preprocesador fit SOLO en train
    prep = TabularPreprocessor(
        user_key_col=user_key_col,
        business_key_col=business_key_col,
        target_col=target_col,
    ).fit(df_train)

    train_data = prep.transform(df_train, include_target=True)
    val_data = prep.transform(df_val, include_target=True)

    train_ds = ReviewDataset(train_data, has_target=True)
    val_ds = ReviewDataset(val_data, has_target=True)

    train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True, num_workers=0)
    val_loader = DataLoader(val_ds, batch_size=batch_size, shuffle=False, num_workers=0)

    # Dimensiones
    review_dim = train_data["review_x"].shape[1]
    user_dim = train_data["user_x"].shape[1]
    biz_dim = train_data["biz_x"].shape[1]
    num_users = len(prep.user2idx)
    num_businesses = len(prep.biz2idx)

    model = ReviewRatingNet(
        num_users=num_users,
        num_businesses=num_businesses,
        review_dim=review_dim,
        user_dim=user_dim,
        biz_dim=biz_dim,
        user_emb_dim=32,
        biz_emb_dim=32,
        tower_out_dim=64,
        fusion_hidden_dims=(128, 64),
        dropout=0.15,  
        num_classes=5,
    )

    device = torch.device("cuda") # Cambiadlo si teneis GPU
    model = train_model(
        model=model,
        train_loader=train_loader,
        val_loader=val_loader,
        device=device,
        epochs=epochs,
        lr=lr,
        weight_decay=weight_decay,
        patience=5,
        pos_weight=pos_weight,
    )

    # The evaluate function in build_and_train also needs the pos_weight
    # The signature of evaluate expects pw, which is 'pos_weight' here
    # The current evaluate_model also expects a 'target' key which is 'y' in ReviewDataset
    mae, rmse, cm = evaluate_model(model, val_loader, device, pos_weight) # Pass pos_weight
    print(f"MAE: {mae:.4f}")
    print(f"RMSE: {rmse:.4f}")
    print("Confusion Matrix (1-5 estrellas):")
    print(cm)
    return model, prep


# ============================================================
# 9) Inference
# ============================================================
@torch.no_grad()
def predict_df(
    model: nn.Module,
    prep: TabularPreprocessor,
    df: Any,
    user_key_col: str = "user_id",
    business_key_col: str = "business_id",
    target_col: str = "stars",
    batch_size: int = 512,
) -> pd.DataFrame:
    """
    df debe tener la misma estructura que train antes del merge:
    - columnas de review
    - user_id
    - business_id
    """
    df = to_pandas(df)

    # Para inferencia, si no pasan users/businesses ya mergeados,
    # debes llamar a prepare_merged_df antes.
    data = prep.transform(df, include_target=(target_col in df.columns))
    ds = ReviewDataset(data, has_target=(target_col in df.columns))
    loader = DataLoader(ds, batch_size=batch_size, shuffle=False, num_workers=0)

    device = next(model.parameters()).device
    model.eval()

    preds = []
    probs_all = []

    for batch in loader:
        batch = batch_to_device(batch, device)
        logits = model(
            batch["user_id"],
            batch["business_id"],
            batch["review_x"],
            batch["user_x"],
            batch["biz_x"],
        )
        prob = torch.sigmoid(logits).cpu().numpy()
        pred = corn_predict(logits).cpu().numpy()

        probs_all.append(prob)
        preds.append(pred)

    preds = np.concatenate(preds)
    probs_all = np.concatenate(probs_all)

    out = df.copy().reset_index(drop=True)
    out["pred_stars"] = preds
    out["p_gt_1"] = probs_all[:, 0]
    out["p_gt_2"] = probs_all[:, 1]
    out["p_gt_3"] = probs_all[:, 2]
    out["p_gt_4"] = probs_all[:, 3]
    return out

from sklearn.metrics import mean_absolute_error, mean_squared_error, confusion_matrix
import numpy as np
import torch

def evaluate_model(model, data_loader, device, pos_weight):
    model.eval()

    all_preds = []
    all_targets = []

    with torch.no_grad():
        for batch in data_loader:
            review_x = batch["review_x"].to(device)
            user_x = batch["user_x"].to(device)
            biz_x = batch["biz_x"].to(device)
            user_id = batch["user_id"].to(device)
            biz_id = batch["business_id"].to(device)
            targets = batch["y"].to(device)  # Changed from 'target' to 'y'

            logits = model(user_id, biz_id, review_x, user_x, biz_x)

            # Predicción (clase más probable)
            # CORN prediction handles this differently, not argmax on logits directly
            preds = corn_predict(logits)

            all_preds.extend((preds - 1).cpu().numpy()) # Convert back to 0-4 for consistent comparison
            all_targets.extend(targets.cpu().numpy())

    all_preds = np.array(all_preds)
    all_targets = np.array(all_targets)

    all_preds_1_5 = all_preds + 1
    all_targets_1_5 = all_targets + 1

    mae = mean_absolute_error(all_targets_1_5, all_preds_1_5)
    rmse = mean_squared_error(all_targets_1_5, all_preds_1_5)

    cm = confusion_matrix(all_targets_1_5, all_preds_1_5, labels=[1,2,3,4,5])

    fig, ax = plt.subplots()
    
    im = ax.imshow(cm)

    # Etiquetas de ejes
    classes = [1, 2, 3, 4, 5]
    ax.set_xticks(np.arange(len(classes)))
    ax.set_yticks(np.arange(len(classes)))
    ax.set_xticklabels(classes)
    ax.set_yticklabels(classes)

    ax.set_xlabel("Predicción")
    ax.set_ylabel("Valor real")
    ax.set_title("Matriz de confusión (1-5 estrellas)")

    # Mostrar valores dentro de las celdas
    for i in range(cm.shape[0]):
        for j in range(cm.shape[1]):
            ax.text(j, i, cm[i, j], ha="center", va="center")

    plt.tight_layout()
    plt.show()
    return mae, rmse, cm


In [ ]:

model, prep = build_and_train(
    train=train,
    users=users,
    businesses=businesses,
    user_key_col="user_id",
    business_key_col="business_id",   
    target_col="stars",
    batch_size=128,
    epochs=2,
    lr=3e-3,
    weight_decay=1e-3,
)


In [ ]:

# Inferencia sobre un dataframe con la misma estructura de train
pred_df = predict_df(model, prep, test, business_key_col="business_id")